In [1]:
import os
import torch

os.environ[
    "CUDA_DEVICE_ORDER"
] = "PCI_BUS_ID"  # Arrange GPU devices starting from 0
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Set the GPU 2 to use

In [2]:
import sys

MONTH = '2023_10'
NPY = '2023_10_19_07_21_48_seg_53'
START = '200'
UNTIL = '25'

MONTH = '2023_11'
NPY = '2023_11_02_18_22_28'
START = '35'
UNTIL = '15'

MONTH = '2023_03'
NPY = '2023_03_20_11_55_00_seg_4'
START = '30'
UNTIL = '20'

# MONTH = '2023_09'
# NPY = '2023_09_07_07_33_48_seg_0'
# START = '10'
# UNTIL = '1'
MONTH = '2025_01'
NPY = '2025_01_02_08_03_43_seg_63'
START = '0'
UNTIL = '20'
# 2025_01_02_08_03_43_seg_63
# NPY =  '2023_10_12_11_13_43_seg_8',
# NPY =  '2023_02_27_15_21_49_seg_175_seq_3',
# NPY =  '2023_11_01_07_30_47_seg_241',

sys.argv = [
    '--n_process=1',
    '--src_base_dir',
    f'/media/vol/shared/mars_dataset/ego_motion_temp/',
    '--output_dir',
    '/home/mars/mars_test/',
    '--npy_prefixes',
    f'{NPY}',
    '--s',
    f'{START}',
    '--u',
    f'{UNTIL}',
    '--override_output',
    '--debug',
]

In [3]:
import sys
import argparse
import concurrent.futures
import glob
import logging
import os
import time
import traceback
import multiprocessing as mp

from multiprocessing import Process
from multiprocessing import Queue

# set environment vars before numpy import
from marsdataio.os.setenv import set_env_vars, set_all_random_seed

set_env_vars(1)

import cv2

cv2.setNumThreads(0)

import numpy as np
import torch

from visdom import Visdom

from marsdataio.logging import init_logger, deinit_logger
from marsdataio.os.multiprocess import NoDaemonPool
from dataengine.generator.drivemode.drivemode import DriveModeGeneratorHandler
from dataengine.generator.renderer import RenderHandler
from dataengine.generator.nav.nav import NavGenerator, plot_legs_map

# from dataengine.generator.obstacle.segment_tracker import CameraTrackingHandler
from dataengine.generator.obstacle.obstacle import ObstacleGeneratorHandler
from dataengine.generator.obstacle.radar_assoc import ObstacleRadarAssocHandler
from dataengine.generator.road.road import RoadDataGeneratorHandler
from marsdataio.dbhelper import (
    CameraTopic,
    DbReader,
    extract_cam_timestamps,
    get_all_radar_configs,
    get_db_config,
    get_radar_topic,
    run_segmented_databases,
    trim_db_time,
)
from marsdataio.npyhelper import (
    get_all_cameras_params,
    get_db_and_video_paths,
    extract_nav_leg_graph,
    extract_npy,
)
from marstransform.geotrans import ecef2geodetic
from marsneuralzoo.models.drivenet import DriveNet
from marsneuralzoo.models.segnet import Segnet
from marsneuralzoo.models.yolo import Yolo
from marsneuralzoo.models.yolo_seg import SegmentationYolo
from marsneuralzoo.models.e2emvm import E2emvm

# argparse 설정
parser = argparse.ArgumentParser()

parser.add_argument(
    '--vis',
    required=False,
    help='enable visualisation',
    dest='vis',
    action='store_true',
)
parser.add_argument(
    '--npy_prefixes',
    nargs='+',
    required=False,
    default=[],
    help='npy name prefixes, can be more than one, e.g., 2020_09_07 2020_09_07',
)
parser.add_argument(
    '--npy_list_file',
    required=False,
    help=(
        'text file containing list of npy files to process\n'
        'each line should contain the path to an npy file relative to '
        '--src_base_dir'
    ),
)
parser.add_argument(
    '--src_base_dir',
    required=False,
    help='source base dir',
    default='/media/vol/runs/ego_motion',
)
parser.add_argument(
    '--output_dir',
    required=False,
    help='output base path',
    default='/media/vol/runs/unified_data',
)
parser.add_argument(
    '--n_process',
    required=False,
    help='number of processes',
    default=9,
    type=int,
)
parser.add_argument(
    '--s',
    required=False,
    help='npy start time in second',
    default=0,
    type=int,
)
parser.add_argument(
    '--u',
    required=False,
    help='npy duration in second',
    default=0,
    type=int,
)
parser.add_argument(
    '--no_obstacle',
    required=False,
    help='exclude obstacles from data generation',
    dest='no_obstacle',
    action='store_true',
)
parser.add_argument(
    '--no_laneline',
    required=False,
    help='exclude laneline from data generation',
    dest='no_laneline',
    action='store_true',
)
parser.add_argument(
    '--no_drivemode',
    required=False,
    help='exclude drive mode from data generation',
    dest='no_drivemode',
    action='store_true',
)
parser.add_argument(
    '--no_nav',
    required=False,
    help='exclude navigation from data generation',
    dest='no_nav',
    action='store_true',
)
parser.add_argument(
    '--override_output',
    required=False,
    help='override the npy if exist, else skip the datagen if the npy exists',
    dest='override_output',
    action='store_true',
)
parser.add_argument(
    '--debug',
    required=False,
    help='set this flag to enable debug output',
    dest='debug',
    action='store_true',
)
parser.set_defaults(vis=False, override_output=False, debug=False)

# 인자 파싱
options = parser.parse_args()

configs = {
    'db_base_path': '/media/vol/shared/db?',
    'src_base_path': options.src_base_dir,
    'output_base_path': options.output_dir,
    'diskcache_dir': './cache',
    'visualisation': options.vis,
    'remote_vis': False,
    'x_range': 200,
    'lane_model_path': '/media/vol/shared/runs/model/drivenet/drivenet_v100_286_best.pt',
    'seg_model_path': '/media/vol/shared/runs/model/segnet/comma10k/segnet_v112_60.pt',
    'obj_det_model_path': '/media/vol/shared/runs/detection_run/model/yolov3/marsdeepdrive/yolov3_v100_100_last_608.pt',
    'ego_mask_dir': '/media/vol/shared/obstacle/ego_mask/latest',
    'cuboid_prior_wlh': np.array([2.0, 4.0, 1.5]),
    'path_width': 4.0,
    'n_subprocesses': 4,
    'debug': options.debug,
    'gen_obstacle': not options.no_obstacle,
    'gen_laneline': not options.no_laneline,
    'gen_drivemode': not options.no_drivemode,
    'gen_nav': not options.no_nav,
    'osm_psql_params': {
        'dbname': 'osm',
        'user': 'osmuser',
        'password': 'osmuser',
        'host': '1.deep.local.marsauto.io',
        'port': '5432',
    },
    'valhalla_url': 'http://1.deep.local.marsauto.io:8002',
    'start_time': options.s,
    'duration': options.u,
}
# mp.set_start_method('spawn')
session = time.strftime('%Y-%m-%d_%H_%M_%S')
error_log_path = os.path.join(
    configs['output_base_path'], f'error_{session}.log'
)

# Init error logger to be used in child processes.
# logger_queue = init_logger(
#     stdout_lvl=None,
#     file_cfg=(logging.ERROR, error_log_path),
#     enqueue=True,
# )
# Init stdout logger
init_logger(reset_logger=False)

npy_files = []
for npy_prefix in options.npy_prefixes:
    # search inside the `YYYY-MM` directory
    str_month = f'{npy_prefix[:7]}'
    npy_files.extend(
        glob.glob(f'{options.src_base_dir}/{str_month}*/{npy_prefix}*.npy')
    )
    # search inside the `src_base_dir` directory
    npy_files.extend(glob.glob(f'{options.src_base_dir}/{npy_prefix}*.npy'))

if options.npy_list_file:
    with open(options.npy_list_file, 'r') as f:
        npy_list = f.readlines()
        # ignore lines that start with `#`
        npy_list = [npy.strip() for npy in npy_list if npy[0] != '#']
        npy_list = [f'{options.src_base_dir}/{npy}' for npy in npy_list]
        npy_files.extend(npy_list)

# npy_name -> npy_path
npy_files = {os.path.split(npy_file)[1]: npy_file for npy_file in npy_files}

if not options.override_output:
    exists_files = {}
    for npy_name, npy_file in npy_files.items():
        if os.path.exists(f'{options.output_dir}/{npy_name}'):
            exists_files[npy_name] = npy_file

    npy_names = npy_files.keys() - exists_files.keys()
    npy_files = {npy_name: npy_files[npy_name] for npy_name in npy_names}
    if exists_files:
        logging.warning(
            f'Skip {len(exists_files)} npy(s) (already generated), use the'
            f' `--override_output` option to regenerate again.'
        )


npy_files = list(npy_files.values())
if len(npy_files) == 0:
    logging.error(
        f'No matching npy is found for npy_prefixes={options.npy_prefixes} '
        f'or files specified in npy_list_file={options.npy_list_file}'
    )

args = [(configs, npy_file) for npy_file in npy_files]
configs, npy_path = args[0]

set_all_random_seed(0)
np.set_printoptions(precision=3, suppress=True)

vis = Visdom() if configs['remote_vis'] else None
out_dir_path = configs['output_base_path']
npy_filename = os.path.basename(npy_path)
npy_name = os.path.splitext(npy_filename)[0]
out_log_path = os.path.join(out_dir_path, f'{npy_name}.log')

# init_logger(
#     stdout_lvl=logging.INFO,
#     file_cfg=(logging.DEBUG, out_log_path),
#     logger_queue=logger_queue,
# )

npy_data = np.load(npy_path)

db_paths, _ = get_db_and_video_paths(npy_data, configs['db_base_path'])
db = DbReader(db_paths[0])
tname = str(npy_data['vehicle_info']['tname'])

s_time = configs['start_time']
duration = configs['duration']

sensors = npy_data['sensors']
cams_params = get_all_cameras_params(sensors)
main_cam_topic = CameraTopic.search_primary_driving_topic(cams_params.keys())
main_cam_params = cams_params[main_cam_topic]
db_configs = get_db_config(db)
radar_cfgs = get_all_radar_configs(db_configs, tname)
main_radar_topic = get_radar_topic(db)
device = torch.device('cuda')

try:
    db_s_time, duration = trim_db_time(
        db, s_time, duration, npy_data, look_ahead_dist=0
    )
except ValueError as e:
    # logging.error(f'skip {npy_name} due to {e}')
    exit()


obj_seg_model = SegmentationYolo(
    device=device,
    model_path='/media/vol/shared/runs/model/yolo11/v0/best.torchscript',
)
e2emvm = E2emvm(multiview=False)

kj/filesystem-disk-unix.c++:1690: warning: PWD environment variable doesn't match current directory; pwd = /home/mars


[2025-01-22 14:48:45,780] [WARNING] _utils.py: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
Loaded SuperPoint model


In [4]:
import logging
import traceback
import os
import cv2
import numpy as np
import torch
import itertools
from collections import defaultdict
from itertools import count
from typing import Dict, List, Optional
from lapsolver import solve_dense
from scipy.sparse import csr_matrix
from sknetwork.clustering import Leiden

from dataengine.generator.obstacle.debug_utils import draw_mask
from dataengine.generator.obstacle.assignment import assign_segments
from dataengine.generator.obstacle.mask import (
    load_ego_masks,
    remove_ego_body_from_masks,
    remove_overlapped_area,
)

from marsdataio.os.cache import cleanup_cache, create_ordered_dict
from marsneuralzoo.models.yolo_seg import SegmentationYolo
from marsneuralzoo.models.e2emvm import E2emvm
from marsdataio.dbhelper import (
    extract_image,
    collate_images,
    DbTopic,
    DbHandler,
)
from marsdataio.npyhelper import get_all_cameras_params
from marsdataio.videowriter import VideoWriter
from marsdataio.npyrenderer.renderer import generate_colors


def dfs(graph, node, visited, component):
    visited.add(node)
    component.add(node)
    for neighbor in graph[node].keys():  # 2차원 구조에서 인접 노드 가져오기
        if neighbor not in visited:
            dfs(graph, neighbor, visited, component)


class ObstaclePoint:
    id_counter = count()

    def __init__(
        self,
    ):
        self.id = next(self.id_counter)
        self.cam_to_uv: Dict[int, Dict[int, np.ndarray]] = {}
        self.cam_to_nuv: Dict[int, Dict[int, np.ndarray]] = {}
        self.cam_to_kpt_idx: Dict[int, Dict[int, int]] = {}


class PointGraph:
    def __init__(self):
        self._cam_to_kpt_to_op_id = defaultdict(lambda: defaultdict(dict))
        self._time_to_op_id_set = defaultdict(set)
        self._op_id_to_op_dict: Dict[int, ObstaclePoint] = {}

    def add_match(
        self,
        time0,
        cam_idx0,
        kpt_idx0,
        uv0,
        nuv0,
        time1,
        cam_idx1,
        kpt_idx1,
        uv1,
        nuv1,
    ):
        op_id0 = self._cam_to_kpt_to_op_id[time0][cam_idx0].get(kpt_idx0, None)
        op_id1 = self._cam_to_kpt_to_op_id[time1][cam_idx1].get(kpt_idx1, None)

        op = None
        if op_id0 is None and op_id1 is None:
            op = ObstaclePoint()
            self._op_id_to_op_dict[op.id] = op

            op.cam_to_uv.setdefault(time0, {})[cam_idx0] = uv0
            op.cam_to_nuv.setdefault(time0, {})[cam_idx0] = nuv0

            op.cam_to_uv.setdefault(time1, {})[cam_idx1] = uv1
            op.cam_to_nuv.setdefault(time1, {})[cam_idx1] = nuv1

            self._cam_to_kpt_to_op_id[time0][cam_idx0][kpt_idx0] = op.id
            self._cam_to_kpt_to_op_id[time1][cam_idx1][kpt_idx1] = op.id

        elif op_id0 is not None and op_id1 is None:
            op = self._op_id_to_op_dict[op_id0]

            op.cam_to_uv.setdefault(time1, {})[cam_idx1] = uv1
            op.cam_to_nuv.setdefault(time1, {})[cam_idx1] = nuv1

            self._cam_to_kpt_to_op_id[time1][cam_idx1][kpt_idx1] = op.id

        elif op_id0 is None and op_id1 is not None:
            op = self._op_id_to_op_dict[op_id1]

            op.cam_to_uv.setdefault(time0, {})[cam_idx0] = uv0
            op.cam_to_nuv.setdefault(time0, {})[cam_idx0] = nuv0

            self._cam_to_kpt_to_op_id[time0][cam_idx0][kpt_idx0] = op.id

        else:
            op = self._op_id_to_op_dict[op_id0]

        self._time_to_op_id_set[time0].add(op.id)
        self._time_to_op_id_set[time1].add(op.id)


class SegmentTracklet:
    """
    Track information of segment. Contains it's unique id and measurements from
    camera. Uses camera id (timestamp and index) as a key and mask from camera
    will be a value. Predicted mask should be updated when track is missed.
    """

    id_counter = count()

    def __init__(self):
        self.id = next(self.id_counter)
        # cam_to_segment_mask[timestamp][cam_index] -> mask
        self.cam_to_segment_mask: Dict[int, Dict[int, np.ndarray]] = {}

        self.missed_count = 0
        self.predicted_mask: np.ndarray = None
        self.merged = False

    def add_cam_segment_mask(
        self, timestamp: int, cam_index: int, segment_mask: np.ndarray
    ):
        """Add new segment mask detected from camera.
        :param timestamp: timestamp
        :param cam_index: cam_index
        :param segment_mask: segment mask
        """
        self.cam_to_segment_mask.setdefault(timestamp, {})[
            cam_index
        ] = segment_mask

        self.missed_count = 0
        self.predicted_mask = None

    def update_predicted_mask(self, mask):
        """
        When tracking fails, use the predicted mask to track in the next frame
        """
        self.predicted_mask = mask

    def query_segment_mask(self, timestamp: int, cam_index: int):
        """query segment mask detected from camera.
        :param timestamp: timestamp
        :param cam_index: cam_index
        """
        if timestamp not in self.cam_to_segment_mask:
            raise KeyError(
                f'Timestamp {timestamp} not found in cam_to_segment_mask'
            )

        if cam_index not in self.cam_to_segment_mask[timestamp]:
            raise KeyError(
                f'Cam index {cam_index} not found for timestamp {timestamp}'
            )

        return self.cam_to_segment_mask[timestamp][cam_index]

    def track_count(self):
        return len(self.cam_to_segment_mask)


class SegmentTracker:
    """
    This class tracks the segments detected by segment model using optical flow.
    """

    def __init__(
        self,
        cam_index: int,
        id_tracklet_dict: Dict[int, SegmentTracklet],
        max_missed_track=12,
        min_iou_thresh=0.3,
    ):
        """Initialises states with tracklet_database."""
        self.cam_index = cam_index
        self.prev_timestamp = -1
        self.prev_gray: Optional[np.ndarray] = None  # H x W

        # id_tracklet_dict[id] -> tracklet
        self.id_tracklet_dict = id_tracklet_dict

        # missed_tracklets[id] -> tracklet
        self.missed_tracklets: Dict[int, SegmentTracklet] = {}

        # tracked_tracklets[id] -> tracklet
        self.tracked_tracklets: Dict[int, SegmentTracklet] = {}
        self.grid = None

        self.max_missed_track = max_missed_track
        self.min_iou_thresh = min_iou_thresh

    def track(self, timestamp, image, segs):
        """Takes a new camera measurements and tracks currently activated
        segment tracklets.
        :param timestamp: timestamp
        :param image: image
        :param segs: segment masks from deep model
        :return: recent tracklet ids.
        """
        h, w = image.shape[:2]
        shape = image.shape[:2]
        curr_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        curr_gray = cv2.resize(curr_gray, (w // 2, h // 2))

        if self.prev_gray is None:
            self.prev_timestamp = timestamp
            self.prev_gray = curr_gray

            h, w = image.shape[:2]
            remap = np.meshgrid(
                np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32)
            )
            self.grid = np.stack(remap, axis=2)

        flow = self._calculate_flow(self.prev_gray, curr_gray)
        flow = cv2.resize(flow, (w, h)) * 2
        warping_matrix = self.grid - flow

        # project active tracklets' ids into image
        tracked_id_image = np.full(shape, -1.0, dtype=np.float32)
        missed_id_image = np.full(shape, -1.0, dtype=np.float32)

        for id, seg_trl in self.tracked_tracklets.items():
            seg = seg_trl.query_segment_mask(
                self.prev_timestamp, self.cam_index
            )
            tracked_id_image[seg > 0] = id

        for id, seg_trl in self.missed_tracklets.items():
            seg = seg_trl.predicted_mask
            missed_id_image[seg > 0] = id

        tracked_id_image = cv2.remap(
            tracked_id_image,
            warping_matrix,
            None,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REFLECT,
        )

        missed_id_image = cv2.remap(
            missed_id_image,
            warping_matrix,
            None,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REFLECT,
        )

        tracked_id_image = tracked_id_image.astype(int)
        tracked_ids = np.unique(tracked_id_image.flatten())
        tracked_ids = tracked_ids[tracked_ids > -1]
        tracked_ids = list(tracked_ids)

        missed_id_image = missed_id_image.astype(int)
        missed_ids = np.unique(missed_id_image.flatten())
        missed_ids = missed_ids[missed_ids > -1]
        missed_ids = list(missed_ids)

        prev_ids = tracked_ids + missed_ids

        segs_len = len(prev_ids)
        expected_segs = np.zeros((segs_len, h, w), dtype=np.uint8)
        missed_offset = 0
        for b, id in enumerate(tracked_ids):
            id_seg = tracked_id_image == id
            expected_segs[b] = id_seg
            missed_offset += 1

        for b, id in enumerate(missed_ids):
            id_seg = missed_id_image == id
            expected_segs[missed_offset + b] = id_seg

        half_exp_segs = expected_segs[:, ::2, ::2]
        half_segs = segs[:, ::2, ::2]

        (
            matched_idx_pair,
            missed_idx,
            unmatched_seg_idx,
            c_wise_false_pair,
            r_wise_false_pair,
        ) = assign_segments(half_exp_segs, half_segs, self.min_iou_thresh)

        for r, c in c_wise_false_pair:
            keep = segs[c]
            remove = expected_segs[r]

            inter = (remove & keep).astype(bool)
            segs[c][inter] = 0

        for r, c in r_wise_false_pair:
            keep = expected_segs[r]
            remove = segs[c]

            inter = (remove & keep).astype(bool)
            remove[inter] = 0

        out_tracklet_ids = []
        self.missed_tracklets = {}
        self.tracked_tracklets = {}
        for id_idx, seg_idx in matched_idx_pair:
            id = prev_ids[id_idx]

            seg = segs[seg_idx]
            seg_trl = self.id_tracklet_dict[id]

            seg_trl.add_cam_segment_mask(timestamp, self.cam_index, seg)
            out_tracklet_ids.append(id)
            self.tracked_tracklets[id] = seg_trl

        for id_idx in missed_idx:
            id = prev_ids[id_idx]
            seg_trl = self.id_tracklet_dict[id]
            seg_trl.missed_count += 1

            if seg_trl.missed_count < self.max_missed_track:
                self.missed_tracklets[id] = seg_trl
                seg_trl.predicted_mask = expected_segs[id_idx]

        for seg_idx in unmatched_seg_idx:
            seg = segs[seg_idx]
            seg_trl = SegmentTracklet()
            self.id_tracklet_dict[seg_trl.id] = seg_trl
            seg_trl.add_cam_segment_mask(timestamp, self.cam_index, seg)
            out_tracklet_ids.append(seg_trl.id)

            self.tracked_tracklets[seg_trl.id] = seg_trl

        self.prev_timestamp = timestamp
        self.prev_gray = curr_gray

        return out_tracklet_ids

    def _calculate_flow(self, src_gray, dst_gray):
        flow = cv2.calcOpticalFlowFarneback(
            src_gray, dst_gray, None, 0.5, 4, 30, 3, 5, 1.2, 0
        )
        return flow


class SegmentMatcher:
    """
    This class matches the segments detected using the feature matching model.
    """

    def __init__(
        self,
        cam_params: Dict[str, any],
        cam_to_trl_ids_dict: Dict[int, Dict[int, List[int]]],
        trl_id_to_trl_dict: Dict[int, SegmentTracklet],
        target_pairs,
        epipolar_thresholds,
        out_debug_dir=None,
    ):
        self._cam_params = list(cam_params.values())
        self.img_wh = self._cam_params[0].img_wh
        # _cam_to_trl_ids_dict[timestamp][cam_index] -> list[tracklet_id]
        self._cam_to_trl_ids_dict = cam_to_trl_ids_dict
        # _trl_id_to_trl_dict[id] -> tracklet
        self._trl_id_to_trl_dict = trl_id_to_trl_dict

        # TODO make parameters configurable!
        self._minimum_matched_kpt_count = 4
        self._target_pairs = target_pairs
        self._target_Ec1c0s = {}
        self._epipolar_thresholds = {}

        # prepare essential matrixes and epipolr thresholds
        for i, (fidx0, fidx1) in enumerate(self._target_pairs):
            key = f'{fidx0}_{fidx1}'

            idx0 = fidx0 - 7
            idx1 = fidx1 - 7

            T_bc0 = self._cam_params[idx0].T_bc
            T_bc1 = self._cam_params[idx1].T_bc
            R_bc0 = T_bc0[:3, :3]
            P_bc0 = T_bc0[:3, 3]
            R_bc1 = T_bc1[:3, :3]
            P_bc1 = T_bc1[:3, 3]

            R_c1b = R_bc1.T
            P_c1b = -R_c1b @ P_bc1

            R_c1c0 = R_c1b @ R_bc0
            P_c1c0 = R_c1b @ P_bc0 + P_c1b
            x, y, z = P_c1c0
            P_skew = np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])
            Ec1c0 = P_skew @ R_c1c0
            self._target_Ec1c0s[key] = Ec1c0
            self._epipolar_thresholds[key] = epipolar_thresholds[i]

        # prepare debugs
        self._random_colors = generate_colors()
        self.out_debug_dir = out_debug_dir
        if self.out_debug_dir is not None:
            os.makedirs(self.out_debug_dir, exist_ok=True)
            self._match_video_writers = {}
            n_cams = len(self._cam_params)
            for idx0, idx1 in self._target_pairs:
                key = f'{idx0-n_cams}_{idx1-n_cams}'
                match_video_path = os.path.join(
                    out_debug_dir, f"match_{key}.mp4"
                )

                self._match_video_writers[key] = cv2.VideoWriter(
                    match_video_path,
                    cv2.VideoWriter_fourcc(*'mp4v'),
                    20,
                    (
                        self._cam_params[0].img_wh[0] * 2,
                        self._cam_params[0].img_wh[1],
                    ),
                )

    def match(
        self,
        timestamp,
        features,
        undistorted_kpts_list,
        preds,
        images: List[np.ndarray] = [],
        frame_count: int = -1,
    ):
        """
        :param timestamp: timestamp
        :param images: lits of images from 7 cams
        :param frame_count: frame count for debug
        :return
            segment_id_pair: segment_id_pair[cam_idx0][cam_idx1]->list[(id0,id1,score)]

        """

        all_matches = preds['matches']

        # segment ids when keypoints are projected into id_image
        kpt_seg_ids = []

        pre_computed_seg_area = {}

        # undistort keypoints for epipolar constraints and collect segment ids
        for cam_idx, kpts in enumerate(features['keypoints']):
            seg_ids = np.full(kpts.shape[0], -1, dtype=int)

            shape = (self.img_wh[1], self.img_wh[0])
            id_image = np.full(shape, -1.0, dtype=np.float32)

            tracked_trl_ids = self._cam_to_trl_ids_dict[timestamp][cam_idx]
            for id in tracked_trl_ids:
                seg_trl = self._trl_id_to_trl_dict[id]
                seg = seg_trl.query_segment_mask(timestamp, cam_idx)
                id_image[seg > 0] = id
                pre_computed_seg_area[(id, timestamp, cam_idx)] = np.sum(seg)

            for idx, kpt in enumerate(kpts):
                x, y = kpt.astype(int)
                seg_id = id_image[y, x]
                if seg_id >= 0:
                    seg_ids[idx] = seg_id

            kpt_seg_ids.append(seg_ids)

        # find same segment tracklet between camera pairs
        segment_id_pair = {}
        n_cams = len(self._cam_params)
        for fidx0, fidx1 in self._target_pairs:
            key = f'{fidx0}_{fidx1}'

            idx0 = fidx0 - n_cams
            idx1 = fidx1 - n_cams

            segment_id_pair[f'{idx0}_{idx1}'] = []

            matches = all_matches[f'matches{fidx0}_{key}'][0].cpu().numpy()
            confs = all_matches[f'conf_scores_{key}'][0, :, 0].cpu().numpy()

            seg_ids0 = kpt_seg_ids[idx0]
            seg_ids1 = kpt_seg_ids[idx1]

            valid_indices = np.flatnonzero(
                (matches >= 0) & (confs >= 0.02) & (seg_ids0 >= 0)
            )
            kpts0 = features['keypoints'][idx0]
            kpts0_idxs = np.array(range(kpts0.shape[0]), int)

            kpts1 = features['keypoints'][idx1]
            kpts1_idxs = np.array(range(kpts1.shape[0]), int)

            kpts0 = kpts0[valid_indices]
            kpts0_idxs = kpts0_idxs[valid_indices]
            seg_ids0 = seg_ids0[valid_indices]
            undists0 = undistorted_kpts_list[idx0][valid_indices]

            kpts1_idx = matches[valid_indices]
            kpts1 = kpts1[kpts1_idx]
            kpts1_idxs = kpts1_idxs[kpts1_idx]
            seg_ids1 = seg_ids1[kpts1_idx]
            undists1 = undistorted_kpts_list[idx1][kpts1_idx]

            valid_indices = np.flatnonzero(seg_ids1 >= 0)

            kpts0 = kpts0[valid_indices]
            kpts0_idxs = kpts0_idxs[valid_indices]
            seg_ids0 = seg_ids0[valid_indices]
            undists0 = undists0[valid_indices]

            kpts1 = kpts1[valid_indices]
            kpts1_idxs = kpts1_idxs[valid_indices]
            seg_ids1 = seg_ids1[valid_indices]
            undists1 = undists1[valid_indices]

            E_c1c0 = self._target_Ec1c0s[key]

            undists0 = undists0[:, :, np.newaxis]
            temp = E_c1c0 @ undists0

            undists1 = undists1[:, np.newaxis, :]
            epipolar_constrains = (undists1 @ temp).flatten()

            epipolar_constrains = (
                np.abs(epipolar_constrains) < self._epipolar_thresholds[key]
            )

            before_epi0 = kpts0
            kpts0 = kpts0[epipolar_constrains]
            kpts0_idxs = kpts0_idxs[epipolar_constrains]
            seg_ids0 = seg_ids0[epipolar_constrains]

            before_epi1 = kpts1
            kpts1 = kpts1[epipolar_constrains]
            kpts1_idxs = kpts1_idxs[epipolar_constrains]
            seg_ids1 = seg_ids1[epipolar_constrains]

            unique_ids0 = np.unique(seg_ids0.flatten())
            unique_ids1 = np.unique(seg_ids1.flatten())

            id_idx0 = {}
            for idx, id in enumerate(unique_ids0):
                id_idx0[id] = idx

            id_idx1 = {}
            for idx, id in enumerate(unique_ids1):
                id_idx1[id] = idx

            rows = unique_ids0.shape[0]
            cols = unique_ids1.shape[0]

            # construct hit matrix to get best match
            hit_mat = np.zeros((rows, cols))

            for id0, id1 in zip(seg_ids0, seg_ids1):
                r = id_idx0[id0]
                c = id_idx1[id1]
                hit_mat[r, c] += 1

            # reject match when more than two segments are mached with one segment
            for i in range(hit_mat.shape[0]):
                if np.sum(hit_mat[i] > self._minimum_matched_kpt_count) > 1:
                    hit_mat[i] = 0

            for j in range(hit_mat.shape[1]):
                if np.sum(hit_mat[:, j] > self._minimum_matched_kpt_count) > 1:
                    hit_mat[:, j] = 0

            matched_indices = np.array(solve_dense(-hit_mat)).T
            matches = []

            for m in matched_indices:
                if hit_mat[m[0], m[1]] > self._minimum_matched_kpt_count:
                    seg_id0 = unique_ids0[m[0]]
                    seg_id1 = unique_ids1[m[1]]

                    seg0_area = pre_computed_seg_area[
                        (seg_id0, timestamp, idx0)
                    ]
                    seg1_area = pre_computed_seg_area[
                        (seg_id1, timestamp, idx1)
                    ]

                    score = hit_mat[m[0], m[1]] ** 2 / (seg0_area * seg1_area)

                    matches.append(
                        (
                            unique_ids0[m[0]],
                            unique_ids1[m[1]],
                            score,
                        )
                    )

            segment_id_pair[f'{idx0}_{idx1}'] = matches

            # save match video
            if self.out_debug_dir is not None:
                self.render_debug_video(
                    timestamp,
                    frame_count,
                    images,
                    idx0,
                    idx1,
                    before_epi0,
                    before_epi1,
                    kpts0,
                    kpts1,
                    matches,
                )

        return segment_id_pair

    def cleanup(self):
        if self.out_debug_dir is not None:
            for _, writer in self._match_video_writers.items():
                writer.release()

    def render_debug_video(
        self,
        timestamp,
        frame_count,
        images,
        idx0,
        idx1,
        before_epi0,
        before_epi1,
        kpts0,
        kpts1,
        matches,
    ):
        key = f'{idx0}_{idx1}'
        img0 = images[idx0].copy()
        img1 = images[idx1].copy()

        for i, (id0, id1, _) in enumerate(matches):
            seg0 = self._trl_id_to_trl_dict[id0]
            mask0 = seg0.query_segment_mask(timestamp, idx0)
            draw_mask(img0, mask0, i, 0.8)
            seg1 = self._trl_id_to_trl_dict[id1]
            mask1 = seg1.query_segment_mask(timestamp, idx1)
            draw_mask(img1, mask1, i, 0.8)

        stitched_img = np.concatenate([img0, img1], axis=1)
        vis_img = stitched_img.copy()

        vis_lines_before = np.concatenate(
            [before_epi0, before_epi1], axis=1
        ).astype(np.int32)
        vis_lines_before[:, 2] += img0.shape[1]

        vis_lines = np.concatenate([kpts0, kpts1], axis=1).astype(np.int32)
        vis_lines[:, 2] += img0.shape[1]

        for i, line in enumerate(vis_lines_before):
            c = (0, 0, 0)
            line = line.tolist()
            pt1, pt2 = (line[0], line[1]), (line[2], line[3])
            cv2.line(vis_img, pt1, pt2, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt1, radius=4, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt2, radius=4, color=c, lineType=cv2.LINE_AA)

        for i, line in enumerate(vis_lines):
            c = self._random_colors[i % len(self._random_colors)]
            line = line.tolist()
            pt1, pt2 = (line[0], line[1]), (line[2], line[3])
            cv2.line(vis_img, pt1, pt2, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt1, radius=4, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt2, radius=4, color=c, lineType=cv2.LINE_AA)
        cv2.putText(
            vis_img,
            f"{frame_count}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            1,
            cv2.LINE_AA,
        )

        self._match_video_writers[key].write(vis_img)


class CameraTrackingHandler(DbHandler):
    def __init__(
        self,
        npy_data,
        obj_seg_model: SegmentationYolo,
        e2emvm: E2emvm,
        cache_dir,
        ego_mask_dir,
        out_debug_dir=None,
    ):
        """Handler to run model inference and tracking for all cameras."""
        self.npy_data = npy_data
        self._db_name = npy_data['db_filename']
        self._vehicle_name = npy_data['vehicle_info']['tname']
        self._cache_dir = cache_dir
        self._frame_count = 0

        # Camera initialization
        sensors = npy_data['sensors']
        self._cam_params = get_all_cameras_params(sensors)
        self._cam_param_list = list(self._cam_params.values())

        # outputs
        # _cam_to_trl_ids_dict[timestamp][cam_index] -> List[tracklet_id]
        self._cam_to_trl_ids_dict: Dict[int, Dict[int, List[int]]] = {}
        # _trl_id_to_trl_dict[tracklet_id]->tracklet
        self._trl_id_to_trl_dict: Dict[int, SegmentTracklet] = {}
        # _op_id_to_vtx_dict[obstacle_point_id]->obstacle
        self._op_id_to_op_dict: Dict[int, ObstaclePoint] = {}

        self._e2e_cache = create_ordered_dict(cache_dir)
        self._time_to_undist: Dict[int, List[List[np.ndarray]]] = {}

        # Prepare segment tracking
        self._seg_model = obj_seg_model
        self._ego_masks = load_ego_masks(ego_mask_dir, self._vehicle_name)
        self._segment_trackers = [
            SegmentTracker(cam_idx, self._trl_id_to_trl_dict)
            for cam_idx in range(len(self._cam_params))
        ]

        # Prepare segment matching
        # TODO make parameters configurable!
        self._cam_match_pairs = [
            # front
            (0 + 7, 1 + 7),
            (0 + 7, 2 + 7),
            # left
            (2 + 7, 3 + 7),
            (3 + 7, 5 + 7),
            # right
            (2 + 7, 4 + 7),
            (4 + 7, 6 + 7),
        ]
        _epipolar_thresholds = [
            # front    # front
            0.1,  #    # (0, 1),
            0.001,  #    # (0, 2),
            # left     # left
            0.1,  #    # (2, 3),
            0.07,  #    # (3, 5),
            # right    # right
            0.1,  #    # (2, 4),
            0.1,  #    # (4, 6),
        ]
        n_cams = len(self._cam_param_list)
        self._time_match_pairs = [(i, n_cams + i) for i in range(n_cams)]
        self._e2e_match_pairs = self._time_match_pairs + self._cam_match_pairs
        self._e2emvm = e2emvm
        self._pg = PointGraph()

        self._segment_matcher = SegmentMatcher(
            self._cam_params,
            self._cam_to_trl_ids_dict,
            self._trl_id_to_trl_dict,
            self._cam_match_pairs,
            _epipolar_thresholds,
            out_debug_dir,
        )

        # _seg_id_pairs_dict[idx0_idx1]-> List[matched_id_pairs]
        self._seg_id_pairs_dict = {}
        for idx0, idx1 in self._cam_match_pairs:
            key = f'{idx0-n_cams}_{idx1-n_cams}'
            self._seg_id_pairs_dict[key] = []

        # Prepare low level feature tracking and matching
        # prev superpoint result
        self._prev_descs: List[np.ndarray] = []

        # _cam_to_kpt_to_op_id[time][cam_idx][featuer_idx]->op_id
        self._cam_to_kpt_to_op_id = defaultdict(lambda: defaultdict(dict))
        self._time_to_op_id_set = defaultdict(set)
        self._op_id_pairs = []
        self.prev_timestamp = -1

        # debug option for intermideate results
        self.out_debug_dir = out_debug_dir

        if self.out_debug_dir is not None:
            video_width = -1
            video_height = -1

            self._video_pos_s = []
            for _, cam_param in self._cam_params.items():
                w, h = cam_param.video_pos + cam_param.img_wh
                video_width = max(video_width, w)
                video_height = max(video_height, h)
                self._video_pos_s.append(cam_param.video_pos)

            seg_video_path = os.path.join(self.out_debug_dir, f'segs.mp4')
            self._seg_video_writer = VideoWriter(
                seg_video_path, video_width, video_height, fps=20
            )

            trl_before_matching_video_path = os.path.join(
                self.out_debug_dir, f'before_matching.mp4'
            )
            self._trl_before_matching_video_writer = VideoWriter(
                trl_before_matching_video_path,
                video_width,
                video_height,
                fps=20,
            )

            trl_after_matching_video_dir = os.path.join(
                self.out_debug_dir, f"after_matching_temp.mp4"
            )
            self._trl_after_matching_video_writer = VideoWriter(
                trl_after_matching_video_dir,
                video_width,
                video_height,
                fps=20,
            )

            keypoint_video_dir = os.path.join(
                self.out_debug_dir, f"keypoint.mp4"
            )
            self.keypoint_video_writer = cv2.VideoWriter(
                keypoint_video_dir,
                cv2.VideoWriter_fourcc(*'mp4v'),
                20,
                (
                    self._cam_param_list[0].img_wh[0],
                    self._cam_param_list[0].img_wh[1],
                ),
            )

    def _handle_cam_msg(self, timestamp, seven_cam_image):

        imgs = []
        for _, cam_param in self._cam_params.items():
            img = extract_image(
                seven_cam_image, cam_param.img_wh, cam_param.video_pos
            )
            imgs.append(img)

        seg_results = self._seg_model.segment_batch(imgs)

        self._cam_to_trl_ids_dict[timestamp] = {}
        cam_names = list(self._cam_params.keys())
        for idx, seg_result in enumerate(seg_results):
            seg_result['masks'] = remove_ego_body_from_masks(
                seg_result['masks'], self._ego_masks[cam_names[idx]]
            )
            seg_result['masks'] = remove_overlapped_area(
                seg_result['masks'], seg_result['scores']
            )
            curr_tracklet_ids = self._segment_trackers[idx].track(
                timestamp, imgs[idx], seg_result['masks']
            )
            self._cam_to_trl_ids_dict[timestamp][idx] = curr_tracklet_ids

        curr_torch_imgs, curr_features = self._e2emvm.extract_features(imgs)
        self._e2e_cache[timestamp] = (
            curr_torch_imgs,
            curr_features,
        )

        if self.prev_timestamp < 0:
            self._undistort_keypoints(timestamp, curr_features)
            self.prev_timestamp = timestamp
            if self.out_debug_dir:
                self._render_debug_videos(timestamp, imgs, seg_results)
            return

        prev_torch_imgs, prev_features = self._e2e_cache[self.prev_timestamp]

        torch_imgs = torch.cat([prev_torch_imgs, curr_torch_imgs])
        features = {}
        features['keypoints'] = (
            prev_features['keypoints'] + curr_features['keypoints']
        )
        features['descriptors'] = (
            prev_features['descriptors'] + curr_features['descriptors']
        )
        features['scores'] = prev_features['scores'] + curr_features['scores']

        preds = self._e2emvm(torch_imgs, features, self._e2e_match_pairs)

        undistorted_kpts_list = self._undistort_keypoints(
            timestamp, curr_features
        )
        # match segments between cams

        new_id_pairs = self._segment_matcher.match(
            timestamp,
            curr_features,
            undistorted_kpts_list,
            preds,
            imgs,
            self._frame_count,  # for debugging
        )

        all_matches = preds['matches']
        # print("################## ", self.prev_timestamp, timestamp)

        prev_undists = self._time_to_undist[self.prev_timestamp]
        curr_undists = self._time_to_undist[timestamp]
        np_kpts = []
        for kpts in features['keypoints']:
            np_kpts.append(kpts.cpu().numpy())

        for fidx0, fidx1 in self._e2e_match_pairs:
            idx0 = fidx0 if fidx0 < 7 else fidx0 - 7
            idx1 = fidx1 if fidx1 < 7 else fidx1 - 7

            if fidx0 > 6:
                continue

            key = f'{fidx0}_{fidx1}'
            matches = all_matches[f'matches{fidx0}_{key}'][0].cpu().numpy()
            confs = all_matches[f'conf_scores_{key}'][0, :, 0].cpu().numpy()

            valid_indices = np.flatnonzero((matches >= 0) & (confs >= 0.02))

            kpt_idxs0 = valid_indices
            kpt_idxs1 = matches[valid_indices]
            # print("######################################")
            # print(f"{fidx0} <> {fidx1} : {idx0} <---> {idx1}")
            # print("######################################")
            # print(kpt_idxs0)
            # print("######################################")

            # print(kpt_idxs1)
            # print("######################################")

            # raise
            time0 = self.prev_timestamp if fidx0 < 7 else timestamp
            time1 = self.prev_timestamp if fidx1 < 7 else timestamp
            undist0 = prev_undists if fidx0 < 7 else curr_undists
            undist1 = prev_undists if fidx1 < 7 else curr_undists

            for k in range(len(kpt_idxs0)):
                self._pg.add_match(
                    time0,
                    idx0,
                    kpt_idxs0[k],
                    np_kpts[fidx0][kpt_idxs0[k]],
                    undist0[idx0][kpt_idxs0[k]],
                    time1,
                    idx1,
                    kpt_idxs1[k],
                    np_kpts[fidx1][kpt_idxs1[k]],
                    undist1[idx1][kpt_idxs1[k]],
                )

        n_cams = len(self._cam_param_list)
        for idx0, idx1 in self._cam_match_pairs:
            key = f'{idx0-n_cams}_{idx1-n_cams}'
            self._seg_id_pairs_dict[key].append(new_id_pairs[key])

        self.prev_timestamp = timestamp

        if self.out_debug_dir:

            cam_idx = 1
            debug_img = imgs[cam_idx].copy()
            kpts = curr_features['keypoints'][cam_idx]

            for kpt in kpts:
                ikpt = kpt.astype(int)
                cv2.circle(debug_img, ikpt, 3, (255, 0, 0), -1)
            self.keypoint_video_writer.write(debug_img)

            self._render_debug_videos(timestamp, imgs, seg_results)

    def get_topics(self):
        return [DbTopic.CAM_MSG]

    def __call__(self, timestamp, topic, data):
        try:
            if topic == DbTopic.CAM_MSG:
                self._handle_cam_msg(timestamp, data)
                self._frame_count += 1
        except KeyboardInterrupt:
            raise KeyboardInterrupt
        except Exception as e:
            logging.error(
                f'Exception at {timestamp}, Topic: {topic}, Db: {self._db_name}'
                f'\n{traceback.format_exc()}'
            )
            raise e

    def get_processed_data(self):
        """
        Before returning processed data, this functions handles all previous correspondence matches
        to remove conflict between cameras.
        Firstly, it finds conflictc in each target pairs and remove false-positives.
        Finally, it groups results to generate a unified tracklet
        """

        groups = []
        n_cams = len(self._cam_param_list)
        for fidx0, fidx1 in self._cam_match_pairs:
            idx0 = fidx0 - n_cams
            idx1 = fidx1 - n_cams

            pairs_list = self._seg_id_pairs_dict[f'{idx0}_{idx1}']

            graph01 = defaultdict(lambda: defaultdict(lambda: 0.0))
            graph10 = defaultdict(lambda: defaultdict(lambda: 0.0))

            for pairs in pairs_list:
                if len(pairs) < 1:
                    continue
                for pair in pairs:
                    id0, id1, score = pair
                    curr_score01 = graph01[id0][id1]

                    if score > curr_score01:
                        graph01[id0][id1] = score
                        graph10[id1][id0] = score

            to_remove_edge = []
            id1_pairs = []
            for id0, id1_to_score in graph01.items():
                if len(id1_to_score) < 2:
                    continue

                id1s = list(id1_to_score.keys())
                id1_pairs = list(itertools.combinations(id1s, 2))
                for id1_0, id1_1 in id1_pairs:
                    trl1_0 = self._trl_id_to_trl_dict[id1_0]
                    timestamps0 = set(trl1_0.cam_to_segment_mask.keys())

                    trl1_1 = self._trl_id_to_trl_dict[id1_1]
                    timestamps1 = set(trl1_1.cam_to_segment_mask.keys())

                    conflict = bool(timestamps0 & timestamps1)

                    if not conflict:
                        continue

                    score0_1_0 = graph01[id0][id1_0]
                    score0_1_1 = graph01[id0][id1_1]

                    if score0_1_0 > score0_1_1:
                        to_remove_edge.append((id0, id1_1))
                    else:
                        to_remove_edge.append((id0, id1_0))

            for id0, id1 in to_remove_edge:
                if id0 in graph01:
                    if id1 in graph01[id0]:
                        del graph01[id0][id1]

                    if len(graph01[id0]) == 0:
                        del graph01[id0]
                if id1 in graph10:
                    if id0 in graph10[id1]:
                        del graph10[id1][id0]

                    if len(graph10[id1]) == 0:
                        del graph10[id1]

            to_remove_edge = []
            id0_pairs = []
            for id1, id0_to_score in graph10.items():
                if len(id0_to_score) < 2:
                    continue

                id0s = list(id0_to_score.keys())
                id0_pairs = list(itertools.combinations(id0s, 2))

                for id0_0, id0_1 in id0_pairs:
                    trl0_1 = self._trl_id_to_trl_dict[id0_0]
                    timestamps0 = set(trl0_1.cam_to_segment_mask.keys())

                    trl0_1 = self._trl_id_to_trl_dict[id0_1]
                    timestamps1 = set(trl0_1.cam_to_segment_mask.keys())

                    conflict = bool(timestamps0 & timestamps1)

                    if not conflict:
                        continue

                    score1_0_0 = graph01[id1][id0_0]
                    score1_0_1 = graph01[id1][id0_1]

                    if score1_0_0 > score1_0_1:
                        to_remove_edge.append((id0_1, id1))
                    else:
                        to_remove_edge.append((id0_1, id1))

            for id0, id1 in to_remove_edge:
                if id0 in graph01:
                    if id1 in graph01[id0]:
                        del graph01[id0][id1]

                    if len(graph01[id0]) == 0:
                        del graph01[id0]
                if id1 in graph10:
                    if id0 in graph10[id1]:
                        del graph10[id1][id0]

                    if len(graph10[id1]) == 0:
                        del graph10[id1]

            graph01.update(graph10)

            visited = set()
            for node in graph01:
                if node not in visited:
                    components = set()
                    dfs(graph01, node, visited, components)
                    groups.append(components)

        merged = []
        while groups:
            first = groups.pop(0)
            merged_group = first
            i = 0
            while i < len(groups):
                if merged_group & groups[i]:
                    merged_group |= groups.pop(i)
                    i = 0
                else:
                    i += 1

            merged.append(merged_group)

        for group in merged:
            to_merge_trl = []
            for id in group:
                to_merge_trl.append(self._trl_id_to_trl_dict[id])
            merged_trl = SegmentTracklet()

            for trl in to_merge_trl:

                for (
                    timestamp,
                    cam_idx_to_masks,
                ) in trl.cam_to_segment_mask.items():
                    merged_trl.cam_to_segment_mask.setdefault(
                        timestamp, {}
                    ).update(cam_idx_to_masks)

                del self._trl_id_to_trl_dict[trl.id]

                for (
                    timestamp,
                    cam_idx_to_masks,
                ) in trl.cam_to_segment_mask.items():
                    for cam_idx, _ in cam_idx_to_masks.items():
                        ids = self._cam_to_trl_ids_dict[timestamp][cam_idx]
                        ids.remove(trl.id)

                        if merged_trl.id not in ids:
                            ids.append(merged_trl.id)

            self._trl_id_to_trl_dict[merged_trl.id] = merged_trl
            merged_trl.merged = True
            merged_trl.cam_to_segment_mask = dict(
                sorted(merged_trl.cam_to_segment_mask.items())
            )
        if self.out_debug_dir is not None:
            self._render_merged_debug_videos()

        return {
            'cam_to_trl_ids_dict': self._cam_to_trl_ids_dict,
            'trl_id_to_trl_dict': self._trl_id_to_trl_dict,
            'e2e_cache': self._e2e_cache,
            'time_to_undists': self._time_to_undist,
            'point_graph': self._pg,
        }

    def cleanup(self):
        self._segment_matcher.cleanup()
        if self.out_debug_dir is not None:
            self._trl_before_matching_video_writer.release()
            self._seg_video_writer.release()
            self.keypoint_video_writer.release()
        # cleanup_cache([self._e2e_cache])

    def _render_debug_videos(self, timestamp, imgs, seg_results):
        # record extracted segments
        vis_imgs = []
        for img, seg_result in zip(imgs, seg_results):
            segs = seg_result['masks']
            img = img.copy()
            for idx, seg in enumerate(segs):
                draw_mask(img, seg, idx, 0.8)
            vis_imgs.append(img)

        video_frame = collate_images(vis_imgs, self._video_pos_s)
        cv2.putText(
            video_frame,
            f'{timestamp:.6f} - {self._frame_count}',
            (40, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            2,
            (0, 255, 255),
            3,
            cv2.LINE_AA,
        )
        self._seg_video_writer.write(video_frame)

        # record tracklets before correspondence matching
        vis_imgs = []
        for idx, trl_ids in self._cam_to_trl_ids_dict[timestamp].items():
            img = imgs[idx].copy()
            for id in trl_ids:
                trl = self._trl_id_to_trl_dict[id]
                mask = trl.query_segment_mask(
                    timestamp,
                    idx,
                )
                draw_mask(img, mask, id, 0.8)
            vis_imgs.append(img)

        video_frame = collate_images(vis_imgs, self._video_pos_s)
        cv2.putText(
            video_frame,
            f'{timestamp} - {self._frame_count}',
            (40, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            2,
            (0, 255, 255),
            3,
            cv2.LINE_AA,
        )
        self._trl_before_matching_video_writer.write(video_frame)

        # record origin images for later correspondence matching video
        vis_imgs = []
        for image in imgs:
            image = image.copy()
            vis_imgs.append(image)

        video_frame = collate_images(vis_imgs, self._video_pos_s)
        cv2.putText(
            video_frame,
            f"{timestamp} - {self._frame_count}",
            (40, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            2,
            (0, 255, 255),
            3,
            cv2.LINE_AA,
        )
        self._trl_after_matching_video_writer.write(video_frame)

    def _render_merged_debug_videos(self):
        self._trl_after_matching_video_writer.release()
        temp_trl_after_matching_video_dir = os.path.join(
            self.out_debug_dir, f"after_matching_temp.mp4"
        )

        cap = cv2.VideoCapture(temp_trl_after_matching_video_dir)
        video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        trl_after_matching_video_dir = os.path.join(
            self.out_debug_dir, f"after_matching.mp4"
        )
        self._trl_after_matching_video_writer = VideoWriter(
            trl_after_matching_video_dir, video_width, video_height, fps=20
        )

        cam_params = list(self._cam_params.values())
        # record tracklets after correspondence matching
        for frame_idx, (timestamp, cam_idx_to_trl_ids) in enumerate(
            self._cam_to_trl_ids_dict.items()
        ):
            ret, seven_cam_image = cap.read()
            if not ret:
                raise SystemError("saved video is missing")

            frames = []
            for cam_idx, trl_ids in cam_idx_to_trl_ids.items():
                cam_param = cam_params[cam_idx]
                image = extract_image(
                    seven_cam_image, cam_param.img_wh, cam_param.video_pos
                ).copy()

                for trl_id in trl_ids:
                    trl = self._trl_id_to_trl_dict[trl_id]
                    mask = trl.query_segment_mask(timestamp, cam_idx)
                    draw_mask(image, mask, trl_id, 0.8)

                frames.append(image)

            video_frame = collate_images(frames, self._video_pos_s)

            self._trl_after_matching_video_writer.write(video_frame)

        self._trl_after_matching_video_writer.release()
        os.remove(temp_trl_after_matching_video_dir)

    def _undistort_keypoints(self, timestamp, features):
        undistorted_kpts_list = []
        camparam_list = list(self._cam_params.values())

        for cam_idx in range(len(self._cam_params)):
            features['descriptors'][cam_idx] = (
                features['descriptors'][cam_idx].detach().cpu().numpy()
            )
            features['keypoints'][cam_idx] = (
                features['keypoints'][cam_idx].cpu().numpy().astype(np.float32)
            )

            K = camparam_list[cam_idx].K
            D = camparam_list[cam_idx].distort_coefs
            R = np.eye(3, dtype=np.float32)
            P = np.eye(3, dtype=np.float32)
            kpts = features['keypoints'][cam_idx]
            kpts = kpts.reshape(-1, 1, 2)

            undistorted_kpts = cv2.fisheye.undistortPoints(kpts, K, D, R=R, P=P)
            undistorted_kpts = undistorted_kpts.reshape(-1, 2)
            undistorted_kpts = np.hstack(
                (undistorted_kpts, np.ones((undistorted_kpts.shape[0], 1)))
            )
            undistorted_kpts_list.append(undistorted_kpts)
            self._time_to_undist[timestamp] = undistorted_kpts_list

        self._time_to_undist[timestamp] = undistorted_kpts_list
        return undistorted_kpts_list

out_debug_dir = None

if configs['debug']:
    out_debug_dir = os.path.join(out_dir_path, npy_name)
    os.makedirs(out_debug_dir, exist_ok=True)
print(out_debug_dir)

camera_tracking_handler = CameraTrackingHandler(
    npy_data=npy_data,
    obj_seg_model=obj_seg_model,
    e2emvm=e2emvm,
    cache_dir=configs['diskcache_dir'],
    ego_mask_dir=configs['ego_mask_dir'],
    out_debug_dir=out_debug_dir,
)


run_segmented_databases(db_paths, db_s_time, duration, camera_tracking_handler)

camera_tracking_handler_output = camera_tracking_handler.get_processed_data()
camera_tracking_handler.cleanup()

/home/mars/mars_test/2025_01_02_08_03_43_seg_63


Loading /media/vol/shared/runs/model/yolo11/v0/best.torchscript for TorchScript inference...


[2025-01-22 14:52:55,602] [WARNING] assignment.py: invalid value encountered in ulong_scalars00 / 20.000, timestamp: 4500.1962910080005


In [5]:
def prepare_undist(cam_params):
    out = []
    for cam in cam_params:

        K_undist = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
            cam.K,
            cam.distort_coefs[:4],
            cam.img_wh,
            np.eye(3),
            balance=1.0,
        )
        map_x, map_y = cv2.fisheye.initUndistortRectifyMap(
            cam.K,
            cam.distort_coefs[:4],
            np.eye(3),
            K_undist,
            cam.img_wh,
            cv2.CV_16SC2,
        )
        out.append((K_undist, map_x, map_y))
    return out


class CameraImageHandler(DbHandler):
    def __init__(
        self,
        npy_data,
        cache_dir,
        out_debug_dir=None,
    ):
        """Handler to run model inference and tracking for all cameras."""
        self.npy_data = npy_data
        self._db_name = npy_data['db_filename']
        self._vehicle_name = npy_data['vehicle_info']['tname']
        self._cache_dir = cache_dir
        self._frame_count = 0

        # Camera initialization
        sensors = npy_data['sensors']
        self._cam_params = get_all_cameras_params(sensors)
        self._cam_param_list = list(self._cam_params.values())
        self.time_to_images = {}
        self.undist_params = prepare_undist(self._cam_param_list)
        self.time_to_undist_images = {}

        # debug option for intermideate results
        self.out_debug_dir = out_debug_dir

        if self.out_debug_dir is not None:
            video_width = -1
            video_height = -1

            self._video_pos_s = []
            for _, cam_param in self._cam_params.items():
                w, h = cam_param.video_pos + cam_param.img_wh
                video_width = max(video_width, w)
                video_height = max(video_height, h)
                self._video_pos_s.append(cam_param.video_pos)

            origin_video_path = os.path.join(self.out_debug_dir, f'origin.mp4')
            self._origin_video_writer = VideoWriter(
                origin_video_path, video_width, video_height, fps=20
            )

    def _handle_cam_msg(self, timestamp, seven_cam_image):
        imgs = []

        for _, cam_param in self._cam_params.items():
            img = extract_image(
                seven_cam_image, cam_param.img_wh, cam_param.video_pos
            )
            imgs.append(img)
        self.time_to_images[timestamp] = imgs

        undist_imgs = []
        for i in range(7):
            img_undist = cv2.remap(
                imgs[i],
                self.undist_params[i][1],
                self.undist_params[i][2],
                cv2.INTER_LINEAR,
            )
            undist_imgs.append(img_undist)
        self.time_to_undist_images[timestamp] = undist_imgs

        if self.out_debug_dir:
            frame = collate_images(imgs, self._video_pos_s)
            self._origin_video_writer.write(frame)

    def get_topics(self):
        return [DbTopic.CAM_MSG]

    def __call__(self, timestamp, topic, data):
        try:
            if topic == DbTopic.CAM_MSG:
                self._handle_cam_msg(timestamp, data)
                self._frame_count += 1

        except KeyboardInterrupt:
            raise KeyboardInterrupt
        except Exception as e:
            logging.error(
                f'Exception at {timestamp}, Topic: {topic}, Db: {self._db_name}'
                f'\n{traceback.format_exc()}'
            )
            raise e

    def cleanup(self):

        if self.out_debug_dir is not None:
            self._origin_video_writer.release()
            ts = list(self.time_to_images.keys())

            to_save = np.array(ts)
            np.save('saves_ns.npy', to_save)


camera_image_handler = CameraImageHandler(
    npy_data=npy_data,
    cache_dir=configs['diskcache_dir'],
    out_debug_dir=out_debug_dir,
)

run_segmented_databases(db_paths, db_s_time, duration, camera_image_handler)

camera_image_handler.cleanup()

In [7]:
for _, trl in camera_tracking_handler._trl_id_to_trl_dict.items():
    trl.time_to_op_id_to_cam_idxs = {}
    trl.op_id_to_trackcount = {}

import os
import shutil
import json
from typing import Dict, List

import numpy as np
import cv2
import torch
from scipy.linalg import svd
from scipy.spatial import cKDTree
from marstransform.loctrans import sensor_to_body, body_to_model
from marsdataio.npyhelper import get_imu_params
from marstransform.lietrans import (
    quat_rotate_np,
    make_se3_np,
    inv_se3_np,
    euler_rotate_np,
)
from marsdataio.npyhelper import get_all_cameras_params
from dataengine.generator.obstacle.mask import (
    mask_to_hull,
    hull_to_bbox,
    load_ego_masks,
)
from dataengine.generator.obstacle.assignment import calc_iou_matrix

from sklearn.decomposition import PCA
import obstacle_lso_py_api as obstacle_lso


def triangulate(undists, T_ecs, min_depth=0.1, max_depth=140):
    """
    Perform triangulation to find the 3D point given two undistorted points and the relative pose.

    Parameters:
    - undist0: list[np.ndarray] (3,) - undistorted point in the cameras
    - undist1: list[np.ndarray] (3,) - undistorted point in the cameras
    - T_ecs: list[np.ndarray (4, 4)] - poses (4x4 matrix) of the cameras wrt the first
    - min_depth: float - minimum allowed depth
    - max_depth: float - maximum allowed depth

    Returns:
    - success: bool - True if triangulation is successful and within depth bounds, False otherwise
    - out: np.ndarray (3,) - the triangulated 3D point
    """
    total = len(undists)
    assert total > 1

    # Camera projection matrices
    T_ces = []
    for T_ec in T_ecs:
        T_ces.append(inv_se3_np(T_ec))

    # Build the matrix A for triangulation
    A = np.zeros((2 * total, 4))
    for i in range(total):
        undist = undists[i]
        P = T_ces[i]

        A[2 * i, :] = undist[0] * P[2, :] - undist[2] * P[0, :]
        A[2 * i + 1, :] = undist[1] * P[2, :] - undist[2] * P[1, :]

    # Solve for the homogeneous 3D point using SVD
    _, _, Vt = svd(A)
    homo_point = Vt[-1]  # Last row of V^T

    # Convert to 3D point
    out = homo_point / homo_point[3]

    valid = True
    for T_ce in T_ces:
        t_cx = T_ce @ out

        if t_cx[2] < min_depth or t_cx[2] > max_depth:
            valid = False
            break

    return valid, out[:3]


def apply_se3(p_a, T_ba):
    """Apply SE3 transformation to an arbitrarily shaped array of 3d points.

    :param p_a: Array of 3d points in original frame.
        Last dimension must be of size 3.
    :param T_ba: SE3 matrix from a-frame to b-frame
    :return p_b: Array of 3d points in transformed frame.
    """
    # Store original shape to restore after applying T_ba
    shape = p_a.shape[:-1]
    p_a = p_a.reshape(-1, 3)
    p_b = (T_ba[:3, :3] @ p_a.T).T + T_ba[:3, 3]
    p_b = p_b.reshape(*shape, 3)
    return p_b


def proj_to_model_plane(points_2d, T_mc, K):
    """Project a set of 2D image points in a given camera to model plane"""
    n_points = len(points_2d)
    T_cm = np.linalg.inv(T_mc)
    normal_m = np.array([0, 0, 1])
    normal_c = (T_cm[:3, :3] @ normal_m.T).T
    h = T_mc[2, 3]
    assert h > 0, "Camera has negative height on plane"
    points_img = np.hstack([points_2d, np.ones((n_points, 1))])
    K_inv = np.linalg.inv(K)
    # points_c: xyz-direction along projection from point back to camera center
    # Want to find alpha * points_c, that satisfies the following
    # np.dot(alpha * points_c, normal_c) = -h
    points_c = (K_inv @ points_img.T).T
    alpha = -h / np.dot(points_c, normal_c)
    # Throw away negative projections
    valid_mask = alpha > 0
    points_c *= alpha[:, None]
    points_m = apply_se3(points_c, T_mc)
    return points_m, valid_mask


def calculate_obstacle_origin(pts_bottom_m):
    """
    calculate expected origin of obstacle from points on model plane
    """
    # Pick minimum-r point as anchor
    r = np.linalg.norm(pts_bottom_m, axis=-1)
    i_anch = np.argmin(r)
    pos_anch = pts_bottom_m[i_anch]
    max_offset = np.array([20, 2.5, 4.0])
    roi = (
        (pos_anch - max_offset < pts_bottom_m)
        & (pts_bottom_m < pos_anch + max_offset)
    ).all(axis=-1)
    pts_bottom_m = pts_bottom_m[roi]
    pos_m = np.mean(pts_bottom_m, axis=0)

    # FIXME Add sanity checks here
    # Check that size is not too small/big
    # Check that object does not collide with ego
    return pos_m


def interpolate_missing(timestamps, time_series):
    """
    Perform in-place interpolation on a 1D or 2D time series with missing values

    Note:
    - Missing values are indicated by nan
    """
    valid = np.isfinite(time_series).all(axis=1)
    if valid.sum() == 0:
        # Empty time series. Do nothing
        return time_series
    for i in range(time_series.shape[1]):
        time_series[:, i] = np.interp(
            timestamps, timestamps[valid], time_series[valid, i]
        )
    return time_series


def split_timestamps(timestamps, interval=3):
    """
    split timestamps into small groups with interval
    """
    start_time = np.floor(timestamps.min())
    end_time = np.ceil(timestamps.max())

    sorted_indices = np.argsort(timestamps)
    sorted_timestamps = timestamps[sorted_indices]

    bins = np.arange(start_time, end_time + interval, interval)
    bin_indices = np.digitize(sorted_timestamps, bins) - 1

    bin_change_indices = np.where(np.diff(bin_indices) != 0)[0] + 1
    splits = np.split(sorted_indices, bin_change_indices)

    groups = []
    for group_indices in splits:
        if len(group_indices) > 0:
            groups.append((group_indices[0], group_indices[-1]))

    return groups


def extract_bottom_edges(hull, max_slope=0.5):
    """
    Extracts bottom edge from hulls
    We want to extract M edges from the hull such that:
    1. The bottom-most point of the hull is included in the extracted edges
    2. The extracted edges have slope less than some threshold
    To do this, we go forward/backward starting from the bottom-most point
    If the slope is not too large, we include the next point
    If the slope is too large, we stop
    """
    edges = np.full_like(hull, -1)
    hull = hull[np.isfinite(hull).all(axis=1)]
    n = len(hull)
    # Roll hull so that bottom point is first
    idx_bottom = np.nanargmax(hull[:, 1])
    hull = np.roll(hull, -idx_bottom, axis=0)
    assert np.nanargmax(hull[:, 1]) == 0
    # Pad end of hull with bottom point
    hull = np.concatenate([hull, hull[0][None]], axis=0)
    diff = np.diff(hull, axis=0)
    slope = np.abs(diff[:, 1] / (diff[:, 0] + 1e-6))
    idx_nodes = np.where(slope > max_slope)[0]
    if len(idx_nodes) >= 2:
        valid_points = np.concatenate(
            [
                hull[idx_nodes[-1] + 1 :],
                hull[: idx_nodes[0] + 1],
            ],
            axis=0,
        )
        edges[: len(valid_points)] = valid_points
    else:
        # Not enough nodes to section hull
        # Fall back to using bottom anchor point
        edges[0] = hull[idx_bottom]
        edges[1] = hull[idx_bottom]
    return edges


def project_points_to_line(points, p0, v):
    # p - p0 계산 (브로드캐스팅 사용)
    diff = points - p0  # shape: (n, 3)
    # t 계산 (스칼라 곱 / 방향 벡터 크기의 제곱)
    t = np.dot(diff, v)  # shape: (n,)
    # 직선 방정식에 t 대입
    projected_points = p0 + t[:, np.newaxis] * v  # shape: (n, 3)
    return projected_points


def make_cuboid(wlh):
    l2 = wlh[1] / 2
    w2 = wlh[0] / 2
    h = wlh[2]

    corners = np.array(
        [
            [l2, w2, 0],
            [l2, -w2, 0],
            [-l2, -w2, 0],
            [-l2, w2, 0],
            [l2, w2, h],
            [l2, -w2, h],
            [-l2, -w2, h],
            [-l2, w2, h],
        ]
    )
    return corners


class ObstacleMeasurements:
    def __init__(
        self,
    ) -> None:
        self.id = -1
        self.wlh = np.empty(
            3,
        )
        self.timestamps = np.empty((0))
        self.t_eos = np.empty((0, 3))
        # yaw from the closest ego model plane
        self.yaw_mos = np.empty((0, 1))

        # timestamp, cam_idx, normalized_bbox
        self.bboxes = np.empty((0, 6))
        self.raw_bboxes = np.empty((0, 6))
        # timestamp, op_id, cam_idx, nuv
        self.op_meas = np.empty((0, 6))


class SolverConfig:
    def __init__(self):
        self.min_seg_pixels = 15
        self.projection_margin = 10
        self.min_bbox_count = 10
        self.min_movement_distance = 2
        self.wlh_priors = np.array(
            [
                [1.8, 4.5, 1.5],
                [1.8, 4.5, 3.0],
                [1.8, 7.2, 3.0],
                [1.8, 9.0, 3.0],
            ]
        )

        # lso configs
        self.cost_to_scale = {}
        self.cost_to_scale['bbox'] = 832
        self.cost_to_scale['reproj'] = 8
        self.cost_to_scale['translation'] = 500
        self.cost_to_scale['yaw'] = 1000


class ObstacleSolver:
    def __init__(
        self,
        npy_data,
        ego_mask_dir,
        e2e_mvm,
        camera_tracking_ouput,
        cache_dir,
        out_debug_dir=None,
    ):
        """Handler to run model inference and tracking for all cameras."""
        self._npy_data = npy_data
        self._db_name = npy_data['db_filename']
        self._vehicle_name = npy_data['vehicle_info']['tname']
        self._ego_masks = load_ego_masks(ego_mask_dir, self._vehicle_name)
        self._e2e_mvm = e2e_mvm
        self._cache_dir = cache_dir
        self._out_debug_dir = out_debug_dir

        # Camera initialization
        sensors = npy_data['sensors']
        self._cam_params = list(get_all_cameras_params(sensors).values())

        _, imu_params = get_imu_params(sensors)
        self.T_bcs = []
        for cam_param in self._cam_params:
            T_bc = sensor_to_body(imu_params.T_mi, cam_param.T_mc)
            self.T_bcs.append(T_bc)

        t_body_to_model = imu_params.T_mi[:3, 3]
        self.T_mb = body_to_model(t_body_to_model)
        self.T_bm = inv_se3_np(self.T_mb)

        self.T_mcs = []
        for T_bc in self.T_bcs:
            T_mc = self.T_mb @ T_bc
            self.T_mcs.append(T_mc)

        self.T_ecs = {}
        self.T_ebs = {}
        self.T_ems = {}

        self._img_wh = self._cam_params[0].img_wh

        self.cam_to_trl_ids_dict = camera_tracking_ouput['cam_to_trl_ids_dict']
        self.trl_id_to_trl_dict = camera_tracking_ouput['trl_id_to_trl_dict']
        self.e2e_cache = camera_tracking_ouput['e2e_cache']
        self.time_to_undists = camera_tracking_ouput['time_to_undists']
        self.pg = camera_tracking_ouput['point_graph']

        self._config = SolverConfig()
        self.max_constant_velocity_time = 3

        self.ob_id_to_mea = {}
        self.rejected_ob_id_to_mea = {}

        self._solved_outs = {}

        self.initialize_ego_data()
        self.match_temporal()
        self.preprocess()
        self.solve()

    def initialize_ego_data(self):
        frames_data = self._npy_data['frames_data']
        ego_poses = frames_data['ecef_pos']
        ego_quats = frames_data['ecef_quat']
        timestamps = frames_data['timestamp']

        # triangulate keypoints observed in sidecameras
        for timestamp, _ in self.pg._time_to_op_id_set.items():
            frame_idx = np.searchsorted(timestamps, timestamp)

            ego_pose = ego_poses[frame_idx]
            ego_quat = ego_quats[frame_idx]

            T_eb = make_se3_np(quat_rotate_np(ego_quat), ego_pose)
            self.T_ebs[timestamp] = T_eb
            self.T_ems[timestamp] = T_eb @ self.T_bm

            T_ecs = []
            for cam_idx in range(len(self._cam_params)):
                T_bc = self.T_bcs[cam_idx]
                T_ec = T_eb @ T_bc
                T_ecs.append(T_ec)
            self.T_ecs[timestamp] = T_ecs

    def match_temporal(self):
        total_times = list(self.T_ecs.keys())
        search_span = 4

        checked_match = set()
        for _, trl in self.trl_id_to_trl_dict.items():
            # if trl.id != 887:
            #     continue

            overlapped_info = []

            for i, (time, cam_idx_to_dict) in enumerate(
                trl.cam_to_segment_mask.items()
            ):
                if len(cam_idx_to_dict) > 1:
                    if 5 in cam_idx_to_dict:
                        overlapped_info.append((total_times.index(time), 5))
                    if 6 in cam_idx_to_dict:
                        overlapped_info.append((total_times.index(time), 6))

            for overlapped_time_idx, overlapped_cam_idx in overlapped_info:
                start = max(overlapped_time_idx - search_span, 0)
                end = min(overlapped_time_idx + search_span, len(total_times))

                time0 = total_times[overlapped_time_idx]
                torch_image0, feature0 = self.e2e_cache[time0]
                undists0 = self.time_to_undists[time0]

                cam_idx0 = 3
                cam_idx1 = 5

                if overlapped_cam_idx == 6:
                    cam_idx0 = 4
                    cam_idx1 = 6

                target_pairs = []
                idx_to_time = {}
                idx_to_time[0] = time0
                target_idx = 1

                torch_img_list = [torch_image0[cam_idx0]]

                features = {}
                features['keypoints'] = [
                    feature0['keypoints'][cam_idx0],
                ]
                features['descriptors'] = [
                    feature0['descriptors'][cam_idx0],
                ]
                features['scores'] = [
                    feature0['scores'][cam_idx0],
                ]

                undists = [undists0[cam_idx0]]

                for i in range(start, end):
                    time1 = total_times[i]

                    key = (time0, cam_idx0, time1, cam_idx1)
                    if key in checked_match:
                        continue

                    checked_match.add(key)

                    torch_image1, feature1 = self.e2e_cache[time1]
                    undists1 = self.time_to_undists[time1]

                    torch_img_list.append(torch_image1[cam_idx1])

                    features['keypoints'].append(
                        feature1['keypoints'][cam_idx1]
                    )
                    features['descriptors'].append(
                        feature1['descriptors'][cam_idx1]
                    )
                    features['scores'].append(feature1['scores'][cam_idx1])

                    undists.append(undists1[cam_idx1])

                    target_pairs.append((0, target_idx))
                    idx_to_time[target_idx] = time1
                    target_idx += 1

                torch_imgs = torch.stack(torch_img_list, dim=0)
                preds = self._e2e_mvm(torch_imgs, features, target_pairs)

                np_kpts = []
                for kpts in features['keypoints']:
                    np_kpts.append(kpts.cpu().numpy())

                all_matches = preds['matches']
                for idx0, idx1 in target_pairs:
                    key = f'{idx0}_{idx1}'
                    matches = all_matches[f'matches0_{key}'][0].cpu().numpy()
                    confs = (
                        all_matches[f'conf_scores_{key}'][0, :, 0].cpu().numpy()
                    )

                    valid_indices = np.flatnonzero(
                        (matches >= 0) & (confs >= 0.02)
                    )

                    kpt_idxs0 = valid_indices
                    kpt_idxs1 = matches[valid_indices]
                    for k in range(len(kpt_idxs0)):
                        self.pg.add_match(
                            idx_to_time[idx0],
                            cam_idx0,
                            kpt_idxs0[k],
                            np_kpts[idx0][kpt_idxs0[k]],
                            undists[idx0][kpt_idxs0[k]],
                            idx_to_time[idx1],
                            cam_idx1,
                            kpt_idxs1[k],
                            np_kpts[idx1][kpt_idxs1[k]],
                            undists[idx1][kpt_idxs1[k]],
                        )

    def preprocess(self):
        """
        collect measurnets for LSE
        calculate initial poses and orientation of obstacles
        """
        timestamps_in_scope = np.array(list(self.T_ems.keys()))
        T_ems = np.array(list(self.T_ems.values()))
        t_ems = T_ems[:, :3, 3]

        ego_pos_tree = cKDTree(t_ems)

        pca = PCA(n_components=1)
        # collect measurements for LSE
        for trl_id, trl in self.trl_id_to_trl_dict.items():
            # print(trl_id)
            mea = ObstacleMeasurements()

            mea.id = trl_id

            mea_timestamps = []
            mea_t_eos = []
            mea_yaw_mos = []
            mea_bboxes = []
            mea_raw_bboxes = []
            mea_op_meas = []
            # estimate origin of obstacle from bottom edge
            # if there is information for triangulated points, use it for initial yaw
            for time, cam_idx_to_mask in trl.cam_to_segment_mask.items():
                # check valid bbox

                t_eo = np.full(3, np.nan)
                bottom_pts_m = []
                valid_raw_bboxes = []
                valid_bboxes = []

                for cam_idx, mask in cam_idx_to_mask.items():
                    if cam_idx == 3 or cam_idx == 5:
                        for op_id in self.pg._time_to_op_id_set[time]:
                            op: ObstaclePoint = self.pg._op_id_to_op_dict[op_id]
                            if len(op.cam_to_uv) < 3:
                                continue
                            cam_idx_to_uv = op.cam_to_uv.get(time, None)
                            if cam_idx_to_uv is None:
                                continue

                            uv0 = cam_idx_to_uv.get(cam_idx, None)
                            if uv0 is None:
                                continue

                            iuv0 = uv0.astype(int)
                            if mask[iuv0[1], iuv0[0]] > 0:
                                op_mea = np.zeros(6)
                                op_mea[0] = op_id
                                op_mea[1] = time
                                op_mea[2] = cam_idx
                                op_mea[3:] = op.cam_to_nuv[time][cam_idx]
                                mea_op_meas.append(op_mea)

                    hull = mask_to_hull(mask)
                    if hull.shape[0] < 3:
                        continue

                    bbox = hull_to_bbox(hull).astype(
                        float
                    )  # left top right bottom

                    bbox_is_valid = (
                        bbox[0] > self._config.projection_margin
                        and bbox[1] > self._config.projection_margin
                        and bbox[2]
                        < self._img_wh[0] - self._config.projection_margin
                        and bbox[3]
                        < self._img_wh[1] - self._config.projection_margin
                        and (bbox[2] - bbox[0]) > self._config.min_seg_pixels
                        and (bbox[3] - bbox[1]) > self._config.min_seg_pixels
                    )

                    if cam_idx == 2:

                        mask = self._ego_masks['frontCamera2State']
                        ibbox = bbox.astype(int)

                        bbox_is_valid = (
                            (not mask[ibbox[1], ibbox[0]])
                            and (not mask[ibbox[3], ibbox[0]])
                            and (not mask[ibbox[1], ibbox[2]])
                            and (not mask[ibbox[3], ibbox[2]])
                            and bbox_is_valid
                        )

                    if bbox_is_valid:
                        # add bbox
                        raw_bbox = np.zeros((6,))
                        raw_bbox[0] = time
                        raw_bbox[1] = cam_idx
                        raw_bbox[2:] = bbox.flatten()

                        valid_raw_bboxes.append(raw_bbox)

                        bbox = bbox.reshape(-1, 1, 2)
                        undists = cv2.fisheye.undistortPoints(
                            bbox,
                            self._cam_params[cam_idx].K,
                            self._cam_params[cam_idx].distort_coefs,
                        )

                        undist_bbox = np.zeros((6,))
                        undist_bbox[0] = time
                        undist_bbox[1] = cam_idx
                        undist_bbox[2:] = undists.flatten()
                        valid_bboxes.append(undist_bbox)

                    bbox = bbox.flatten()
                    edges = None
                    if (
                        bbox[3]
                        < self._img_wh[1] - self._config.projection_margin
                    ):
                        edges = extract_bottom_edges(hull)

                    if edges is not None:
                        K = self._cam_params[cam_idx].K
                        D = self._cam_params[cam_idx].distort_coefs
                        edges = edges.reshape(-1, 1, 2).astype(np.float32)
                        u_pts_c = cv2.fisheye.undistortPoints(
                            edges, K, D
                        ).reshape(-1, 2)
                        T_mc = self._cam_params[cam_idx].T_mc
                        pts_m, valid = proj_to_model_plane(
                            u_pts_c, T_mc, K=np.eye(3, 3)
                        )
                        pts_m = pts_m[valid]
                        if pts_m.shape[0] > 0:
                            bottom_pts_m.append(pts_m)

                if len(bottom_pts_m) > 0:
                    bottom_pts_m = [pts for pts in bottom_pts_m if len(pts) > 0]
                    bottom_pts_m = np.vstack(bottom_pts_m)

                    pos_m = calculate_obstacle_origin(bottom_pts_m)

                    T_em = self.T_ems[time]
                    t_eo = apply_se3(pos_m, T_em)

                # op_id_to_cam_idxs = trl.time_to_op_id_to_cam_idxs.get(time, {})
                # for op_id, cam_idxs in op_id_to_cam_idxs.items():
                #     op = self.pg._op_id_to_op_dict[op_id]
                #     if len(op.cam_to_uv) < 50:
                #         continue
                #     for cam_idx in cam_idxs:
                #         op_mea = np.zeros(6)
                #         op_mea[0] = op_id
                #         op_mea[1] = time
                #         op_mea[2] = cam_idx
                #         op_mea[3:] = op.cam_to_nuv[time][cam_idx]
                #         mea_op_meas.append(op_mea)

                yaw_mo = 0.0
                mea_raw_bboxes += valid_raw_bboxes
                mea_bboxes += valid_bboxes
                mea_timestamps.append(time)
                mea_t_eos.append(t_eo)
                mea_yaw_mos.append(yaw_mo)

            if len(mea_timestamps) > 0:
                mea.timestamps = np.array(mea_timestamps)
                mea.t_eos = np.array(mea_t_eos)
                mea.yaw_mos = np.array(mea_yaw_mos).reshape(-1, 1)
                mea.bboxes = np.array(mea_bboxes)
                mea.raw_bboxes = np.array(mea_raw_bboxes)
                mea.op_meas = np.array(mea_op_meas)
                self.ob_id_to_mea[trl_id] = mea

        # initial guess filling and smoothing
        for _, mea in self.ob_id_to_mea.items():
            mea.t_eos = interpolate_missing(mea.timestamps, mea.t_eos)

            if np.any(np.isnan(mea.t_eos)):
                continue

            if mea.t_eos.shape[0] < 5:
                continue

            min_t_eos = np.min(mea.t_eos, axis=0)
            max_t_eos = np.max(mea.t_eos, axis=0)
            if (
                np.linalg.norm(max_t_eos - min_t_eos)
                > self._config.min_movement_distance
            ):
                pca.fit(mea.t_eos)
                direction_vec = pca.components_[0]
                cp = np.median(mea.t_eos, axis=0)
                mea.t_eos = project_points_to_line(mea.t_eos, cp, direction_vec)
                mea.t_eos = interpolate_missing(mea.timestamps, mea.t_eos)

            # if mea.t_eos.shape[0] > 15:
            #     cut_off = 2
            #     fs = 50
            #     mea.t_eos[:, 0] = lowpass_filter(mea.t_eos[:, 0], cut_off, fs)
            #     mea.t_eos[:, 1] = lowpass_filter(mea.t_eos[:, 1], cut_off, fs)
            #     mea.t_eos[:, 2] = lowpass_filter(mea.t_eos[:, 2], cut_off, fs)

            # rotations
            groups = split_timestamps(mea.timestamps)
            prev_idx = -1

            for group in groups:

                f_idx = group[0]
                e_idx = group[1]

                if prev_idx > -1:
                    f_idx = prev_idx
                    prev_idx = -1

                vec_e = mea.t_eos[e_idx] - mea.t_eos[f_idx]
                if np.linalg.norm(vec_e) < self._config.min_movement_distance:
                    prev_idx = f_idx
                    continue

                idxs = np.array(range(f_idx, e_idx + 1))

                for time_idx in idxs:
                    curr_t_eo = mea.t_eos[time_idx]

                    _, ego_idx = ego_pos_tree.query(curr_t_eo)
                    curr_time = timestamps_in_scope[ego_idx]
                    curr_T_me = inv_se3_np(self.T_ems[curr_time])
                    curr_R_me = curr_T_me[:3, :3]
                    vec_m = curr_R_me @ vec_e
                    yaw_mo = np.arctan2(vec_m[1], vec_m[0])
                    mea.yaw_mos[time_idx] = yaw_mo

            if prev_idx != -1:
                mea.yaw_mos = interpolate_missing(mea.timestamps, mea.yaw_mos)

            # assign wlh based on initial guess
            iou_scores = np.zeros(self._config.wlh_priors.shape[0])
            iou_counter = 0
            for bbox in mea.bboxes:
                idx = np.searchsorted(mea.timestamps, bbox[0])
                t_eo = mea.t_eos[idx]

                _, ego_idx = ego_pos_tree.query(t_eo)
                ref_time = timestamps_in_scope[ego_idx]
                T_em = self.T_ems[ref_time]
                T_mc = self.T_mcs[int(bbox[1])]
                T_ec = T_em @ T_mc
                T_ce = inv_se3_np(T_ec)

                yaw_mo = mea.yaw_mos[idx, 0]
                R_mo = euler_rotate_np(np.array([0, 0, yaw_mo]))
                R_eo = T_em[:3, :3] @ R_mo

                prior_bboxes = np.zeros((self._config.wlh_priors.shape[0], 4))

                for wlh_idx in range(self._config.wlh_priors.shape[0]):
                    wlh = self._config.wlh_priors[wlh_idx]
                    t_ops = make_cuboid(wlh)

                    t_eps = (R_eo @ t_ops.T).T + t_eo
                    t_cps = apply_se3(t_eps, T_ce)

                    t_cps = t_cps / t_cps[:, 2, np.newaxis]

                    lt = np.min(t_cps[:, :2], axis=0)
                    rb = np.max(t_cps[:, :2], axis=0)

                    prior_bboxes[wlh_idx][:2] = lt
                    prior_bboxes[wlh_idx][2:4] = rb

                mat, _, _ = calc_iou_matrix(
                    prior_bboxes, bbox[2:].reshape(-1, 4)
                )

                iou_scores += mat.flatten()
                iou_counter += 1

            if iou_counter == 0:
                mea.wlh = self._config.wlh_priors[0]
            else:
                best_idx = np.argmax(iou_scores)
                mea.wlh = self._config.wlh_priors[best_idx]

    def check_valid(self, mea: ObstacleMeasurements):
        if len(mea.timestamps) < 5:
            return False

        if not np.all(~np.isnan(mea.t_eos)):
            return False

        if mea.bboxes.shape[0] == 0:
            return False

        if len(np.unique(mea.bboxes[:, 0])) < 15:
            return False

        return True

    def solve(self):
        to_remove = []
        for ob_id, mea in self.ob_id_to_mea.items():
            if not self.check_valid(mea):
                to_remove.append(ob_id)

        for ob_id in to_remove:
            mea = self.ob_id_to_mea[ob_id]
            del self.ob_id_to_mea[ob_id]
            self.rejected_ob_id_to_mea[ob_id] = mea

        T_ems_timestamps = np.array(list(self.T_ems.keys()))
        T_ems = np.array(list(self.T_ems.values()))
        T_mcs = np.array(self.T_mcs)

        obstacle_lso.Init(
            T_ems_timestamps,
            T_ems,
            T_mcs,
            self._config.cost_to_scale,
        )

        solved_outs = {}

        for _, mea in self.ob_id_to_mea.items():
            input = obstacle_lso.Input()
            input.wlh = mea.wlh
            input.timestamps = mea.timestamps
            input.t_eos = mea.t_eos
            input.yaw_mos = mea.yaw_mos
            input.bboxes = mea.bboxes
            input.op_meas = mea.op_meas
            out = obstacle_lso.Solve(input)
            solved_outs[mea.id] = out

        obstacle_lso.Clear()

        if self._out_debug_dir:
            mea_dir = os.path.join(self._out_debug_dir, "meas")
            if os.path.exists(mea_dir):
                shutil.rmtree(mea_dir)
            for _, mea in self.ob_id_to_mea.items():
                # Main processing
                save_obstacle_measurements(mea_dir, mea)
            # Convert T_ems to JSON
            T_ems = np.array(list(self.T_ems.values()))
            T_ems_path = os.path.join(mea_dir, 'T_ems.json')
            with open(T_ems_path, 'w') as f:
                json.dump(T_ems.tolist(), f)

            T_ems_timestamps = np.array(list(self.T_ems.keys()))
            T_ems_timestamps_path = os.path.join(
                mea_dir, 'T_ems_timestamps.json'
            )
            with open(T_ems_timestamps_path, 'w') as f:
                json.dump(T_ems_timestamps.tolist(), f)

            T_mcs = np.array(self.T_mcs)
            T_mcs_path = os.path.join(mea_dir, 'T_mcs.json')
            with open(T_mcs_path, 'w') as f:
                json.dump(T_mcs.tolist(), f)

            lso_config_path = os.path.join(mea_dir, f'lso_config.json')

            with open(lso_config_path, 'w') as f:
                json.dump(self._config.cost_to_scale, f)


def save_obstacle_measurements(dir, mea):
    save_path = os.path.join(dir, f'{mea.id}')

    os.makedirs(save_path, exist_ok=True)
    prior_path = os.path.join(save_path, f'wlh.json')
    time_path = os.path.join(save_path, f'time.json')
    t_eos_path = os.path.join(save_path, f't_eos.json')
    yaw_path = os.path.join(save_path, f'yaw_mos.json')
    bboxes_path = os.path.join(save_path, f'bboxes.json')
    op_mea_path = os.path.join(save_path, f'op_mea.json')

    with open(prior_path, 'w') as f:
        json.dump(mea.wlh.tolist(), f)

    with open(time_path, 'w') as f:
        json.dump(mea.timestamps.tolist(), f)

    with open(t_eos_path, 'w') as f:
        json.dump(mea.t_eos.tolist(), f)

    with open(yaw_path, 'w') as f:
        json.dump(mea.yaw_mos.flatten().tolist(), f)

    with open(bboxes_path, 'w') as f:
        json.dump(mea.bboxes.tolist(), f)

    with open(op_mea_path, 'w') as f:
        json.dump(mea.op_meas.tolist(), f)


ObstaclePoint.id_counter = count()

obstacle_solver = ObstacleSolver(
    npy_data=npy_data,
    ego_mask_dir=configs['ego_mask_dir'],
    e2e_mvm=e2emvm,
    camera_tracking_ouput=camera_tracking_handler_output,
    cache_dir=configs['diskcache_dir'],
)
# raise

Solving... 1
Done.

Solver Summary (v 2.2.0-eigen-(3.4.0)-lapack-suitesparse-(5.7.1)-eigensparse)

                                     Original                  Reduced
Parameter blocks                          801                      801
Parameters                               1603                     1603
Residual blocks                          1413                     1413
Residuals                                6044                     6044

Minimizer                        TRUST_REGION
Trust region strategy                  DOGLEG (TRADITIONAL)
Sparse linear algebra library    SUITE_SPARSE 

                                        Given                     Used
Linear solver          SPARSE_NORMAL_CHOLESKY   SPARSE_NORMAL_CHOLESKY
Threads                                     2                        2
Linear solver ordering              AUTOMATIC                      801

Cost:
Initial                          4.673041e+05
Final                            9.082154e-01
Change  

In [8]:
# obstacle_solver.track_keypoints()
# obstacle_solver.initialize_ego_data()

# obstacle_solver.match_temporal()

# obstacle_solver.assign_keypoints_to_segements()
# obstacle_solver.preprocess()

# obstacle_solver.solve()

In [9]:
import json
import os
import numpy as np


def save_obstacle_measurements(dir, mea):
    save_path = os.path.join(dir, f'{mea.id}')

    os.makedirs(save_path, exist_ok=True)
    prior_path = os.path.join(save_path,f'wlh.json')
    time_path = os.path.join(save_path, f'time.json')
    t_eos_path = os.path.join(save_path, f't_eos.json')
    yaw_path = os.path.join(save_path, f'yaw_mos.json')
    bboxes_path = os.path.join(save_path, f'bboxes.json')
    op_mea_path = os.path.join(save_path, f'op_mea.json')

    with open(prior_path, 'w') as f:
        json.dump(mea.wlh.tolist(), f)

    with open(time_path, 'w') as f:
        json.dump(mea.timestamps.tolist(), f)

    with open(t_eos_path, 'w') as f:
        json.dump(mea.t_eos.tolist(), f)
 
    with open(yaw_path, 'w') as f:
        json.dump(mea.yaw_mos.flatten().tolist(), f)

    with open(bboxes_path, 'w') as f:
        json.dump(mea.bboxes.tolist(), f)

    with open(op_mea_path, 'w') as f:
        json.dump(mea.op_meas.tolist(), f)

!rm -rf mea/*
!rm -rf mea_out/**

mea_dir = 'mea'
for id, mea in obstacle_solver.ob_id_to_mea.items():
    # Main processing
    save_obstacle_measurements(mea_dir, mea)
# Convert T_ems to JSON
T_ems = np.array(list(obstacle_solver.T_ems.values()))
T_ems_path = os.path.join(mea_dir, 'T_ems.json')
with open(T_ems_path, 'w') as f:
    json.dump(T_ems.tolist(), f)

T_ems_timestamps = np.array(list(obstacle_solver.T_ems.keys()))
T_ems_timestamps_path = os.path.join(mea_dir, 'T_ems_timestamps.json')
with open(T_ems_timestamps_path, 'w')as f:
    json.dump(T_ems_timestamps.tolist(), f)

T_mcs = np.array(obstacle_solver.T_mcs)
T_mcs_path = os.path.join(mea_dir, 'T_mcs.json')
with open(T_mcs_path, 'w') as f:
    json.dump(T_mcs.tolist(), f)

# # Convert op_id_to_t_eo to JSON
# op_id_to_t_eo = {}
# op_id_to_op = obstacle_solver.pg._op_id_to_op_dict
# for op_id, op in op_id_to_op.items():
#     if len(op.time_to_t_ep) == 0:
#         continue

#     time_to_t_ep_list = {} 
#     for time, t_eo in op.time_to_t_ep.items():
#         t_eo_list = [t_eo[0], t_eo[1], t_eo[2]]
#         time_to_t_ep_list[time] = t_eo_list

#     op_id_to_t_eo[op_id] = time_to_t_ep_list

# op_ids_path = os.path.join(mea_dir, f'op_id_to_t_eo.json')
# with open(op_ids_path, 'w') as f:
#     json.dump(op_id_to_t_eo, f)

lso_config_path = os.path.join(mea_dir, f'lso_config.json')

with open(lso_config_path, 'w') as f:
    json.dump(obstacle_solver._config.cost_to_scale, f)


In [10]:
from marstransform.lietrans import (
    quat_rotate_np,
    make_se3_np,
    inv_se3_np,
    quat_rotate_np,
    rotate_quat_np,
    euler_rotate_np,
)
from marstransform.loctrans import sensor_to_body
from marsdataio.npyhelper import get_imu_params
import rerun as rr  # pip install rerun-sdk
import rerun.blueprint as rrb
import argparse
from marstransform.lietrans import (
    quat_rotate_np,
    make_se3_np,
    inv_se3_np,
    expm_so3_np,
)
from dataengine.generator.obstacle.obstacle_solver import apply_se3

my_blueprint = rrb.Blueprint(
    rrb.Horizontal(
        rrb.Spatial3DView(
            name='3D', origin='world/body', contents=['world/**']
        ),
        rrb.Horizontal(
            rrb.Vertical(
                rrb.Spatial2DView(
                    name="cam1",
                    origin='world/body/cam1',
                    contents=[
                        'world/body/cam1/**',
                        # 'world/obstacles/**',
                        'world/solved/**',
                    ],
                ),
                rrb.Spatial2DView(
                    name="cam5",
                    origin='world/body/cam5',
                    contents=[
                        'world/body/cam5/**',
                        # 'world/obstacles/**',
                        'world/solved/**',
                    ],
                ),
                rrb.Spatial2DView(
                    name="cam3",
                    origin='world/body/cam3',
                    contents=[
                        'world/body/cam3/**',
                        # 'world/obstacles/**',
                        'world/solved/**',
                    ],
                ),
            ),
            rrb.Vertical(
                rrb.Spatial2DView(
                    name="cam2",
                    origin='world/body/cam2',
                    contents=[
                        'world/body/cam2/**',
                        # 'world/obstacles/**',
                        'world/solved/**',
                    ],
                ),
            ),
            rrb.Vertical(
                rrb.Spatial2DView(
                    name="cam0",
                    origin='world/body/cam0',
                    contents=[
                        'world/body/cam0/**',
                        # 'world/obstacles/**',
                        'world/solved/**',
                    ],
                ),
                rrb.Spatial2DView(
                    name="cam6",
                    origin='world/body/cam6',
                    contents=[
                        'world/body/cam6/**',
                        # 'world/obstacles/**',
                        'world/solved/**',
                    ],
                ),
                rrb.Spatial2DView(
                    name="cam4",
                    origin='world/body/cam4',
                    contents=[
                        'world/body/cam4/**',
                        # 'world/obstacles/**',
                        'world/solved/**',
                    ],
                ),
            ),
        ),
    )
)

NAME = f'{NPY}_{START}_{UNTIL}'
test_recording_id = 'bbb'
rr.init(NAME, spawn=True, recording_id=test_recording_id)
rr.send_blueprint(my_blueprint)

rr.log(
    "world", rr.ViewCoordinates.RIGHT_HAND_Z_DOWN, static=True
)  # Set an up-axis

rr.log(
    "world/xyz",
    rr.Arrows3D(
        vectors=[[10, 0, 0], [0, 10, 0], [0, 0, 10]],
        colors=[[255, 0, 0], [0, 255, 0], [0, 0, 255]],
    ),
    static=True,
)

T_we = next(iter(obstacle_solver.T_ebs.values()))
T_we = inv_se3_np(T_we)


undist_param = camera_image_handler.undist_params

w, h = obstacle_solver._cam_params[0].img_wh
dist = 1.0
body_path = []
for time, T_eb in obstacle_solver.T_ebs.items():
    rr.set_time_seconds('time', time)

    T_wb = T_we @ T_eb

    rr.log(
        'world/body',
        rr.Transform3D(
            translation=T_wb[:3, 3], mat3x3=T_wb[:3, :3], axis_length=1
        ),
    )
    body_path.append(T_wb[:3, 3])

    rr.log(
        f'world/body_path',
        rr.LineStrips3D(
            strips=body_path,
            radii=0.1,
        ),
    )

    rr.log(
        "world/body/point",
        rr.Points3D([0, 0, 0], radii=0.050, colors=[255, 200, 10]),
    )
    undist_images = camera_image_handler.time_to_undist_images[time]
    for i, T_bc in enumerate(obstacle_solver.T_bcs):
        rr.log(
            f'world/body/cam{i}',
            rr.Transform3D(translation=T_bc[:3, 3], mat3x3=T_bc[:3, :3]),
        )
        rr.log(
            f'world/body/cam{i}',
            rr.Pinhole(
                image_from_camera=undist_param[i][0],
                resolution=[w, h],
                image_plane_distance=dist,
            ),
        )

        rr.log(
            f'world/body/cam{i}/bgr',
            rr.Image(undist_images[i], color_model="BGR").compress(
                jpeg_quality=95
            ),
        )


timestamps_in_scope = np.array(list(obstacle_solver.T_ems.keys()))
T_ems = np.array(list(obstacle_solver.T_ems.values()))
t_ems = T_ems[:, :3, 3]
ego_pos_tree = cKDTree(t_ems)


for ob_id, mea in obstacle_solver.ob_id_to_mea.items():
    if len(mea.timestamps) < 2:
        continue

    for i in range(len(mea.timestamps)):

        # if op_id == target_solve_id:
        #     continue
        time = mea.timestamps[i]
        t_eo = mea.t_eos[i]
        yaw_mo = mea.yaw_mos[i, 0]
        if np.isnan(t_eo).all():
            continue

        _, idx = ego_pos_tree.query(t_eo)
        ref_time = timestamps_in_scope[idx]
        T_em = obstacle_solver.T_ems[ref_time]
        T_me = inv_se3_np(T_em)

        # orientation

        R_mo = euler_rotate_np(np.array([0, 0, yaw_mo]))

        R_wo = T_we[:3, :3] @ T_em[:3, :3] @ R_mo
        # center
        t_mo = apply_se3(t_eo, T_me)

        wlh = mea.wlh
        half_xyz = wlh[[1, 0, 2]] / 2
        t_mo[2] += half_xyz[2]
        t_eo = apply_se3(t_mo, T_em)
        t_wo = apply_se3(t_eo, T_we)

        q_wo = rotate_quat_np(R_wo)
        q_wo = rr.Quaternion(xyzw=q_wo[[1, 2, 3, 0]])

        rr.set_time_seconds('time', time)
        rr.log(
            f'world/obstacles/ob{ob_id}/box',
            rr.Boxes3D(
                centers=t_wo,
                half_sizes=half_xyz,
                rotations=q_wo,
                class_ids=ob_id,
                radii=0.05,
            ),
        ),

    cam_idx_to_bbox_last_time = {}
    for i in range(len(mea.bboxes)):
        bbox = mea.bboxes[i]

        time = bbox[0]
        cam_idx = int(bbox[1])
        cam_idx_to_bbox_last_time[cam_idx] = time
        K = np.eye(3)
        D = np.zeros((4,))
        K_undist = camera_image_handler.undist_params[cam_idx][0]

        pts = bbox[2:].reshape(-1, 1, 2)
        pts_3d = np.concatenate(
            (pts, np.ones((pts.shape[0], 1, 1), dtype=pts.dtype)), axis=2
        )

        cam_param = camera_image_handler._cam_param_list[cam_idx]
        rvec = np.array([0.0, 0.0, 0.0])  # Rotation vector
        tvec = np.array([0.0, 0.0, 0.0])  # Translation vector
        # dists, _ = cv2.fisheye.projectPoints(pts_3d, rvec,tvec ,K_undist,np.zeros((4,)))
        dists, _ = cv2.projectPoints(
            pts_3d, rvec, tvec, K_undist, np.zeros((5,))
        )

        dists = dists.flatten()

        rr.set_time_seconds('time', time)
        rr.log(
            f'world/body/cam{cam_idx}/bgr/bbox{ob_id}',
            rr.Boxes2D(
                array=dists,
                array_format=rr.Box2DFormat.XYXY,
                class_ids=ob_id,
            ),
        )

        # rr.set_time_seconds('time', time+ 0.02)
        # rr.log(
        #     f'world/body/cam{cam_idx}/bgr/bbox{op_id}',
        #     rr.Clear(recursive=False)
        # )
    for cam_idx, time in cam_idx_to_bbox_last_time.items():
        rr.set_time_seconds('time', time)
        rr.log(
            f'world/body/cam{cam_idx}/bgr/bbox{ob_id}',
            rr.Clear(recursive=False),
        )

    rr.set_time_seconds('time', mea.timestamps[-1] + 0.02)
    rr.log(f'world/obstacles/ob{ob_id}/box', rr.Clear(recursive=False))

[2025-01-22T06:00:25Z INFO  re_sdk_comms::server] Hosting a SDK server over TCP at 0.0.0.0:9876. Connect with the Rerun logging SDK.
[2025-01-22T06:00:25Z INFO  winit::platform_impl::linux::x11::window] Guessed window scale factor: 2
[2025-01-22T06:00:25Z INFO  re_sdk_comms::server] New SDK client connected from: 127.0.0.1:40328
[2025-01-22T06:00:25Z WARN  wgpu_hal::gles::egl] No config found!
[2025-01-22T06:00:25Z WARN  wgpu_hal::gles::egl] EGL says it can present to the window but not natively
[2025-01-22T06:00:25Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2025-01-22T06:00:25Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2025-01-22T06:00:25Z INFO  egui_wgpu] There were 4 available wgpu adapters: {backend: Vulkan, device_type: DiscreteGpu, name: "NVIDIA GeForce GTX 1080 Ti", driver: "NVIDIA", driver_info: "560.35.03", vendor: 0x10DE, device: 0x1B06}, {backend: Vulkan, device_type: DiscreteGpu, n

In [11]:
import json


def load_output(json_file):
    with open(json_file, 'r') as file:
        data = json.load(file)

    # Parse the data into a dictionary of numpy arrays
    output = {
        "wlh": np.array(data["wlh"]),
        "timestamps": np.array(data["timestamps"]),
        "t_eos": [np.array(t) for t in data["t_eos"]],
        "q_eos": [np.array(q) for q in data["q_eos"]],
        "t_ops": [np.array(q) for q in data["t_ops"]],
    }
    return output


for solved_path in os.listdir("mea_out"):
    c_solved_id = int(os.path.splitext(solved_path)[0])
    c_solved = load_output(os.path.join("mea_out", solved_path))
    c_wlh = c_solved['wlh']
    c_timestamps = c_solved['timestamps']
    c_t_eos = c_solved['t_eos']
    c_q_eos = c_solved['q_eos']
    c_t_ops = c_solved['t_ops']
    t_ops = np.array(c_t_ops)
    t_ops = t_ops[:, 1:4, np.newaxis]

    last_time_solved = 0.0
    t_wcs = []
    for i in range(len(c_timestamps)):
        time = c_timestamps[i]
        last_time_solved = time
        rr.set_time_seconds('time', time)

        t_eo = c_t_eos[i]

        q_eo = c_q_eos[i]

        half_xyz = c_wlh[[1, 0, 2]] / 2

        R_eo = quat_rotate_np(q_eo)

        t_oc = np.array([0, 0, half_xyz[2]])
        t_ec = R_eo @ t_oc + t_eo

        R_wo = T_we[:3, :3] @ R_eo
        q_wo = rotate_quat_np(R_wo)
        q_wo = rr.Quaternion(xyzw=q_wo[[1, 2, 3, 0]])

        t_wc = apply_se3(t_ec, T_we)
        t_wcs.append(t_wc)
        if len(t_wcs) == 60:
            del t_wcs[0]
        rr.log(
            f'world/solved/obs/ob{c_solved_id}',
            rr.Boxes3D(
                centers=t_wc,
                half_sizes=half_xyz,
                rotations=q_wo,
                class_ids=c_solved_id,
                radii=0.05,
            ),
        )
        rr.log(
            f'world/solved/paths/ob{c_solved_id}',
            rr.LineStrips3D(
                strips=t_wcs,
                class_ids=c_solved_id,
                radii=0.05,
            ),
        )

        if t_ops.shape[0] > 5:
            t_eps = (R_eo @ t_ops) + t_eo.reshape(3, 1)

            t_wps = (T_we[:3, :3] @ t_eps) + T_we[:3, 3].reshape(3, 1)
            rr.log(
                f'world/solved/pcl/ob{c_solved_id}',
                rr.Points3D(t_wps, class_ids=c_solved_id),
            )

    rr.set_time_seconds('time', c_timestamps[-1] + 0.02)
    rr.log(f'world/solved/obs/ob{c_solved_id}', rr.Clear(recursive=False))
    rr.log(f'world/solved/pcl/ob{c_solved_id}', rr.Clear(recursive=False))
    rr.log(f'world/solved/paths/ob{c_solved_id}', rr.Clear(recursive=False))

In [12]:
# raise
import obstacle_lso_py_api as obstacle_lso

T_ems_timestamps = np.array(list(obstacle_solver.T_ems.keys()))
T_ems = np.array(list(obstacle_solver.T_ems.values()))
T_mcs = np.array(obstacle_solver.T_mcs)

obstacle_lso.Init(
    T_ems_timestamps, T_ems, T_mcs, obstacle_solver._config.cost_to_scale
)

solved_outs = {}

for _, mea in obstacle_solver.ob_id_to_mea.items():
    print(f"### Solving {mea.id}")
    input = obstacle_lso.Input()
    input.wlh = mea.wlh
    input.timestamps = mea.timestamps
    input.t_eos = mea.t_eos
    input.yaw_mos = mea.yaw_mos
    input.bboxes = mea.bboxes
    input.op_meas = mea.op_meas
    out = obstacle_lso.Solve(input)
    solved_outs[mea.id] = out

obstacle_lso.Clear()

### Solving 3
Solving... 1
Done.

Solver Summary (v 2.2.0-eigen-(3.4.0)-lapack-suitesparse-(5.7.1)-eigensparse)

                                     Original                  Reduced
Parameter blocks                          801                      801
Parameters                               1603                     1603
Residual blocks                          1413                     1413
Residuals                                6044                     6044

Minimizer                        TRUST_REGION
Trust region strategy                  DOGLEG (TRADITIONAL)
Sparse linear algebra library    SUITE_SPARSE 

                                        Given                     Used
Linear solver          SPARSE_NORMAL_CHOLESKY   SPARSE_NORMAL_CHOLESKY
Threads                                     2                        2
Linear solver ordering              AUTOMATIC                      801

Cost:
Initial                          4.673041e+05
Final                            9.08215

In [13]:
for ob_id, output in solved_outs.items():
    c_solved_id = ob_id
    c_wlh = output.wlh
    c_timestamps = output.timestamps
    c_t_eos = output.t_eos
    c_q_eos = output.q_eos
    c_t_ops = output.t_ops
    t_ops = np.array(c_t_ops)
    if t_ops.shape[0] > 0:
        t_ops = t_ops[:, 1:4, np.newaxis]

    last_time_solved = 0.0
    t_wcs = []

    for i in range(len(c_timestamps)):
        time = c_timestamps[i]
        last_time_solved = time
        rr.set_time_seconds('time', time)

        t_eo = c_t_eos[i]

        q_eo = c_q_eos[i]

        half_xyz = c_wlh[[1, 0, 2]] / 2

        R_eo = quat_rotate_np(q_eo)

        t_oc = np.array([0, 0, half_xyz[2]])
        t_ec = R_eo @ t_oc + t_eo

        R_wo = T_we[:3, :3] @ R_eo
        q_wo = rotate_quat_np(R_wo)
        q_wo = rr.Quaternion(xyzw=q_wo[[1, 2, 3, 0]])

        t_wc = apply_se3(t_ec, T_we)
        t_wcs.append(t_wc)
        if len(t_wcs) == 60:
            del t_wcs[0]
        rr.log(
            f'world/solved/obs/ob{c_solved_id}',
            rr.Boxes3D(
                centers=t_wc,
                half_sizes=half_xyz,
                rotations=q_wo,
                class_ids=c_solved_id,
                radii=0.05,
            ),
        )
        rr.log(
            f'world/solved/paths/ob{c_solved_id}',
            rr.LineStrips3D(
                strips=t_wcs,
                class_ids=c_solved_id,
                radii=0.05,
            ),
        )

        if t_ops.shape[0] > 5:
            t_eps = (R_eo @ t_ops) + t_eo.reshape(3, 1)

            t_wps = (T_we[:3, :3] @ t_eps) + T_we[:3, 3].reshape(3, 1)
            rr.log(
                f'world/solved/pcl/ob{c_solved_id}',
                rr.Points3D(t_wps, class_ids=c_solved_id),
            )

    rr.set_time_seconds('time', c_timestamps[-1] + 0.02)
    rr.log(f'world/solved/obs/ob{c_solved_id}', rr.Clear(recursive=False))
    rr.log(f'world/solved/pcl/ob{c_solved_id}', rr.Clear(recursive=False))
    rr.log(f'world/solved/paths/ob{c_solved_id}', rr.Clear(recursive=False))

In [14]:
def check_seg(seg_id: int):
    trl = camera_tracking_handler._trl_id_to_trl_dict[seg_id]
    for time, cam_idx_to_seg in trl.cam_to_segment_mask.items():
        for cam_idx, seg in cam_idx_to_seg.items():
            to_render = camera_image_handler.time_to_images[time][
                cam_idx
            ].copy()
            draw_mask(to_render, seg)
            cv2.imshow("", to_render)
            key = cv2.waitKey(0)
            if key == 27:
                cv2.destroyAllWindows()
                raise
    cv2.destroyAllWindows()


def check_mea(id: int):
    mea = obstacle_solver.ob_id_to_mea[id]

    for i in range(mea.raw_bboxes.shape[0]):
        bbox = mea.raw_bboxes[i]
        time = bbox[0]
        cam_idx = int(bbox[1])

        if cam_idx != 3:
            continue

        to_render = camera_image_handler.time_to_images[time][cam_idx].copy()

        ibbox = bbox[2:].astype(int)
        cv2.rectangle(to_render, ibbox[:2], ibbox[2:], (255, 0, 0), 2)

        bbox = mea.bboxes[i]
        ibbox = bbox[2:].astype(int)

        pts = bbox[2:].reshape(-1, 1, 2)
        pts_3d = np.concatenate(
            (pts, np.ones((pts.shape[0], 1, 1), dtype=pts.dtype)), axis=2
        )

        cam_param = camera_image_handler._cam_param_list[cam_idx]
        rvec = np.array([0.0, 0.0, 0.0])  # Rotation vector
        tvec = np.array([0.0, 0.0, 0.0])  # Translation vector
        dists, _ = cv2.fisheye.projectPoints(
            pts_3d, rvec, tvec, cam_param.K, cam_param.distort_coefs
        )
        idists = dists.astype(int)
        idists = idists.flatten()

        cv2.rectangle(to_render, idists[:2], idists[2:], (0, 0, 255), 2)

        img_undist = cv2.remap(
            to_render,
            camera_image_handler.undist_params[cam_idx][1],
            camera_image_handler.undist_params[cam_idx][2],
            cv2.INTER_LINEAR,
        )

        K_undist = camera_image_handler.undist_params[cam_idx][0]
        undist_images = camera_image_handler.time_to_undist_images[time][
            cam_idx
        ].copy()

        dists, _ = cv2.fisheye.projectPoints(
            pts_3d, rvec, tvec, K_undist, np.zeros((4,))
        )
        idists = dists.astype(int)
        idists = idists.flatten()

        cv2.rectangle(img_undist, idists[:2], idists[2:], (0, 255, 0), 2)

        dists, _ = cv2.projectPoints(
            pts_3d, rvec, tvec, K_undist, np.zeros((5,))
        )
        idists = dists.astype(int)
        idists = idists.flatten()
        cv2.rectangle(img_undist, idists[:2], idists[2:], (0, 255, 255), 2)

        cv2.imshow("1", to_render)
        cv2.imshow("2", img_undist)

        key = cv2.waitKey(0)
        if key == 27:
            cv2.destroyAllWindows()
            raise
    cv2.destroyAllWindows()


# check_seg(578)
cv2.destroyAllWindows()